# LDC LPI yearly composition processing — finalized dictionary

This notebook processes the full georeferenced LPI tall table against the finalized species / functional-group master dictionary and writes **four composition products per year**:

1. top-canopy / first-hit species composition
2. top-canopy / first-hit functional-group composition
3. multilayer / any-hit species composition
4. multilayer / any-hit functional-group composition

Each output contains one row per `PrimaryKey × Year`. `PrimaryKey` is treated as the visit/event identifier, so a genuine revisit in the same year remains a separate row rather than being silently collapsed.

The cover logic mirrors the two `terradactyl::pct_cover()` hit concepts:

- **first hit / top canopy**: one top-down outcome per pin; if `TopCanopy` is blank (`__NO_CANOPY__`), the first hit is the `SoilSurface` observation. Lower canopy layers never substitute for a blank top canopy.
- **any hit / multilayer**: presence at any position from canopy through base. The same taxon or functional group is counted only once per pin, so summed any-hit cover can exceed 100%.

The finalized dictionary is authoritative. `canonical_code` is the downstream taxon identifier, so manual synonym/canonical redirects collapse correctly.

## Visit exclusions and QA flags

- Any `PrimaryKey × Year` containing a `PlantBase` hit is excluded **entirely** from composition processing.
- Remaining visits are retained even if they contain unresolved codes.
- For every retained visit, the notebook calculates the percentage of all protocol LPI hit rows whose dictionary status is `UnknownCode`.
- `unknown_code_gt5pct = True` when those unresolved-code hits exceed 5% of all LPI hit rows in that visit.
- The unknown-code percentage and flag are carried into all four yearly composition outputs and written to a separate QA table.


In [2]:
# ============================================================================
# 1. IMPORTS, PATHS, AND PROCESSING THRESHOLDS
# ============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

try:
    import duckdb
except ImportError:
    raise ImportError(
        "DuckDB is required for the 16M-row processing step.\n"
        "Run: %pip install duckdb"
    )

BASE_DIR = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
)

RAW_LPI_FILE = BASE_DIR / "LDC_LPI_georeferenced_2018_present.csv"

FINAL_MASTER_FILE = (
    BASE_DIR
    / "species_dictionary_outputs"
    / "LDC_LPI_species_dictionary_FINAL_MASTER.csv"
)

OUTPUT_DIR = BASE_DIR / "processed_LPI_composition"

TOP_SPECIES_DIR = OUTPUT_DIR / "top_hit" / "species"
TOP_FG_DIR = OUTPUT_DIR / "top_hit" / "functional_group"
MULTI_SPECIES_DIR = OUTPUT_DIR / "multilayer" / "species"
MULTI_FG_DIR = OUTPUT_DIR / "multilayer" / "functional_group"
QA_DIR = OUTPUT_DIR / "QA"

for p in [
    TOP_SPECIES_DIR,
    TOP_FG_DIR,
    MULTI_SPECIES_DIR,
    MULTI_FG_DIR,
    QA_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

DUCKDB_FILE = OUTPUT_DIR / "LDC_LPI_composition_processing.duckdb"

UNKNOWN_CODE_FLAG_THRESHOLD_PCT = 5.0

assert RAW_LPI_FILE.exists(), RAW_LPI_FILE
assert FINAL_MASTER_FILE.exists(), FINAL_MASTER_FILE

print("Raw LPI:", RAW_LPI_FILE)
print("Final master:", FINAL_MASTER_FILE)
print("Output root:", OUTPUT_DIR)
print("Unknown-code flag threshold:", UNKNOWN_CODE_FLAG_THRESHOLD_PCT, "%")


Raw LPI: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\LDC_LPI_georeferenced_2018_present.csv
Final master: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\species_dictionary_outputs\LDC_LPI_species_dictionary_FINAL_MASTER.csv
Output root: C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present\processed_LPI_composition
Unknown-code flag threshold: 5.0 %


In [3]:
# ============================================================================
# 2. FINAL MASTER QA
# ============================================================================

master = pd.read_csv(
    FINAL_MASTER_FILE,
    low_memory=False
)

assert len(master) == 10960
assert master["observed_code"].nunique(dropna=False) == 10960
assert not master["observed_code"].duplicated().any()

assert master["observed_code"].eq("__NO_CANOPY__").sum() == 1
assert master["observed_code"].eq("NONE").sum() == 1

poar = master.loc[
    master["observed_code"].eq("POAR2R2")
]

assert len(poar) == 1
assert poar.iloc[0]["USDA_accepted_symbol"] == "PONIN"

print("Final master QA: PASS")


Final master QA: PASS


In [9]:
# ============================================================================
# DIAGNOSE STRUCTURALLY MALFORMED RAW LPI CSV ROWS
#
# READ-ONLY:
#   - does not modify the source file
#   - does not write a repaired file
#   - does not drop any rows
#
# Checks:
#   - expected number of fields from header
#   - rows with too few fields
#   - rows with too many fields
#   - captures row number, rid/code/layer where recoverable
# ============================================================================

from pathlib import Path
import csv
import pandas as pd

RAW_LPI_FILE = Path(
    r"C:\NCA_DATA\Vegetation Data\LDC_LPI_2018_present"
    r"\LDC_LPI_georeferenced_2018_present.csv"
)

assert RAW_LPI_FILE.exists(), RAW_LPI_FILE

problems = []

n_rows = 0
n_exact = 0
n_short = 0
n_long = 0

with RAW_LPI_FILE.open(
    "r",
    encoding="utf-8",
    newline="",
    errors="replace",
) as f:

    reader = csv.reader(f)

    header = next(reader)
    expected_cols = len(header)

    print("Expected columns:", expected_cols)
    print("Header:")
    print(header)

    # Helpful positions for diagnostics.
    col_index = {
        name: i
        for i, name in enumerate(header)
    }

    for source_row_number, row in enumerate(
        reader,
        start=2,
    ):

        n_rows += 1
        n_fields = len(row)

        if n_fields == expected_cols:
            n_exact += 1
            continue

        if n_fields < expected_cols:
            n_short += 1
            failure_type = "too_few_fields"
        else:
            n_long += 1
            failure_type = "too_many_fields"

        def safe_value(colname):
            idx = col_index.get(colname)

            if idx is None:
                return None

            if idx >= len(row):
                return None

            return row[idx]

        problems.append(
            {
                "source_row_number": source_row_number,
                "failure_type": failure_type,
                "expected_fields": expected_cols,
                "found_fields": n_fields,
                "field_difference": n_fields - expected_cols,

                "rid": safe_value("rid"),
                "PrimaryKey": safe_value("PrimaryKey"),
                "PlotID": safe_value("PlotID"),
                "PointNbr": safe_value("PointNbr"),
                "layer": safe_value("layer"),
                "code": safe_value("code"),

                # Useful for spotting where truncation/shift occurs.
                "first_5_fields": row[:5],
                "last_5_fields": row[-5:],
            }
        )


problems_df = pd.DataFrame(problems)

print()
print("CSV structural scan complete.")
print(f"Data rows read:          {n_rows:,}")
print(f"Exact-width rows:        {n_exact:,}")
print(f"Too-few-field rows:      {n_short:,}")
print(f"Too-many-field rows:     {n_long:,}")
print(f"Total abnormal rows:     {len(problems_df):,}")

if n_rows > 0:
    print(
        "Percent structurally abnormal:",
        f"{100 * len(problems_df) / n_rows:.8f}%"
    )

if len(problems_df) > 0:
    print("\nProblem-type summary:")
    display(
        problems_df
        .groupby(
            [
                "failure_type",
                "expected_fields",
                "found_fields",
            ],
            dropna=False,
        )
        .size()
        .reset_index(name="n_rows")
        .sort_values(
            "n_rows",
            ascending=False,
        )
    )

    print("\nFirst 100 abnormal rows:")
    display(
        problems_df.head(100)
    )
else:
    print("\nNo structural row-width problems detected.")

Expected columns: 33
Header:
['rid', 'PrimaryKey', 'DBKey.x', 'ProjectKey.x', 'LineKey', 'RecKey', 'layer', 'code', 'chckbox', 'ShrubShape', 'FormType', 'FormDate', 'Direction', 'Measure', 'LineLengthAmount', 'SpacingIntervalAmount', 'SpacingType', 'ShowCheckbox', 'CheckboxLabel', 'PointLoc', 'PointNbr', 'source.x', 'DateLoadedInDb', 'DateVisited', 'DateVisited_parsed', 'SampleYear', 'Latitude_NAD83', 'Longitude_NAD83', 'PlotID', 'ProjectKey.y', 'source.y', 'EcologicalSiteID', 'DBKey.y']

CSV structural scan complete.
Data rows read:          16,150,114
Exact-width rows:        16,150,114
Too-few-field rows:      0
Too-many-field rows:     0
Total abnormal rows:     0
Percent structurally abnormal: 0.00000000%

No structural row-width problems detected.


In [10]:
# ============================================================================
# INGEST RAW LPI THROUGH PYTHON'S VERIFIED CSV PARSER INTO DUCKDB
#
# Python csv.reader has already been verified to parse:
#   16,150,114 data rows
#   33 fields per row
#
# We stream bounded chunks into DuckDB so the entire file never needs to be
# loaded into pandas memory at once.
# ============================================================================

import csv
import pandas as pd

CHUNK_SIZE = 250_000

# Start clean.
con.execute("DROP TABLE IF EXISTS lpi_raw")

rows_buffer = []
rows_ingested = 0
first_chunk = True


with RAW_LPI_FILE.open(
    "r",
    encoding="utf-8",
    newline="",
    errors="replace",
) as f:

    reader = csv.reader(f)

    header = next(reader)

    assert len(header) == 33, (
        f"Expected 33 columns; found {len(header)}."
    )

    for row in reader:

        # We already audited this invariant, but retain it as a hard guard.
        assert len(row) == len(header), (
            f"Unexpected parsed row width: "
            f"{len(row)} vs {len(header)}"
        )

        rows_buffer.append(row)

        if len(rows_buffer) >= CHUNK_SIZE:

            chunk = pd.DataFrame(
                rows_buffer,
                columns=header,
            )

            con.register(
                "lpi_chunk",
                chunk,
            )

            if first_chunk:

                con.execute("""
                    CREATE TABLE lpi_raw AS
                    SELECT *
                    FROM lpi_chunk
                """)

                first_chunk = False

            else:

                con.execute("""
                    INSERT INTO lpi_raw
                    SELECT *
                    FROM lpi_chunk
                """)

            con.unregister("lpi_chunk")

            rows_ingested += len(chunk)

            print(
                f"Ingested {rows_ingested:,} rows"
            )

            rows_buffer.clear()


    # ------------------------------------------------------------------------
    # Final partial chunk
    # ------------------------------------------------------------------------

    if rows_buffer:

        chunk = pd.DataFrame(
            rows_buffer,
            columns=header,
        )

        con.register(
            "lpi_chunk",
            chunk,
        )

        if first_chunk:

            con.execute("""
                CREATE TABLE lpi_raw AS
                SELECT *
                FROM lpi_chunk
            """)

        else:

            con.execute("""
                INSERT INTO lpi_raw
                SELECT *
                FROM lpi_chunk
            """)

        con.unregister("lpi_chunk")

        rows_ingested += len(chunk)

        rows_buffer.clear()


# ============================================================================
# HARD ACCOUNTING CHECK
# ============================================================================

duckdb_count = con.execute("""
SELECT COUNT(*)
FROM lpi_raw
""").fetchone()[0]

print()
print(f"Python-parsed rows: {rows_ingested:,}")
print(f"DuckDB rows:        {duckdb_count:,}")

assert rows_ingested == 16_150_114
assert duckdb_count == 16_150_114

print(
    "PASS: all 16,150,114 verified CSV records "
    "were ingested into DuckDB."
)

Ingested 250,000 rows
Ingested 500,000 rows
Ingested 750,000 rows
Ingested 1,000,000 rows
Ingested 1,250,000 rows
Ingested 1,500,000 rows
Ingested 1,750,000 rows
Ingested 2,000,000 rows
Ingested 2,250,000 rows
Ingested 2,500,000 rows
Ingested 2,750,000 rows
Ingested 3,000,000 rows
Ingested 3,250,000 rows
Ingested 3,500,000 rows
Ingested 3,750,000 rows
Ingested 4,000,000 rows
Ingested 4,250,000 rows
Ingested 4,500,000 rows
Ingested 4,750,000 rows
Ingested 5,000,000 rows
Ingested 5,250,000 rows
Ingested 5,500,000 rows
Ingested 5,750,000 rows
Ingested 6,000,000 rows
Ingested 6,250,000 rows
Ingested 6,500,000 rows
Ingested 6,750,000 rows
Ingested 7,000,000 rows
Ingested 7,250,000 rows
Ingested 7,500,000 rows
Ingested 7,750,000 rows
Ingested 8,000,000 rows
Ingested 8,250,000 rows
Ingested 8,500,000 rows
Ingested 8,750,000 rows
Ingested 9,000,000 rows
Ingested 9,250,000 rows
Ingested 9,500,000 rows
Ingested 9,750,000 rows
Ingested 10,000,000 rows
Ingested 10,250,000 rows
Ingested 10,500,000 

In [15]:
# ============================================================================
# BUILD JOINED LPI TABLE FROM VERIFIED lpi_raw
# ============================================================================

print("Building joined lpi table...")

con.execute("""
CREATE OR REPLACE TABLE lpi AS

WITH raw AS (
    SELECT
        rid,
        PrimaryKey,

        COALESCE(
            NULLIF(TRIM("DBKey.x"), ''),
            NULLIF(TRIM("DBKey.y"), '')
        ) AS DBKey,

        COALESCE(
            NULLIF(TRIM("ProjectKey.x"), ''),
            NULLIF(TRIM("ProjectKey.y"), '')
        ) AS ProjectKey,

        LineKey,
        RecKey,
        layer,
        PointNbr,

        code AS observed_code_raw,

        CASE
            WHEN code IS NULL
                 OR TRIM(code) = ''
            THEN '__NO_CANOPY__'
            ELSE TRIM(code)
        END AS observed_code,

        DateVisited,

        COALESCE(
            TRY_CAST(SampleYear AS INTEGER),
            TRY_CAST(SUBSTR(DateVisited, 1, 4) AS INTEGER)
        ) AS Year,

        TRY_CAST(Latitude_NAD83 AS DOUBLE) AS Latitude_NAD83,
        TRY_CAST(Longitude_NAD83 AS DOUBLE) AS Longitude_NAD83

    FROM lpi_raw
)

SELECT
    r.*,

    m.observed_code AS dictionary_match_code,

    m.resolved_label,
    m.code_class,
    m.resolution_source,

    m.canonical_code,
    m.canonical_label,

    m.dictionary_resolution_status,
    m.hit_interpretation,

    m.USDA_accepted_symbol,
    m.USDA_scientific_name,
    m.USDA_common_name,
    m.USDA_family,

    m.USDA_duration,
    m.USDA_growth_habit,
    m.USDA_native_status_L48,
    m.USDA_trait_status,

    m.MOSAIC_FG_structural,
    m.MOSAIC_FG,
    m.FG_resolution_source,

    CASE
        WHEN m.code_class = 'Plant'
        THEN m.canonical_code
        ELSE NULL
    END AS taxon_id,

    CASE
        WHEN m.code_class = 'Plant'
        THEN m.canonical_label
        ELSE NULL
    END AS taxon_label

FROM raw r

LEFT JOIN species_master m
    ON r.observed_code = m.observed_code
;
""")

print("lpi table created.")

Building joined lpi table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

lpi table created.


## Canonical joined tall table

The raw LPI code is preserved as `observed_code_raw`.

Blank/NA source codes become `__NO_CANOPY__`. Literal `"NONE"` remains a real USDA plant symbol.

For plant rows, `taxon_id` uses the accepted USDA symbol when one exists; otherwise the canonical observed plant code is retained.


In [16]:
# ============================================================================
# 4. JOIN FULL TALL LPI TO FINAL MASTER
#
# IMPORTANT:
#   - lpi_raw was populated through Python csv.reader
#   - all 16,150,114 records were verified and ingested
#   - do NOT read the raw CSV again here
#   - finalized dictionary fields are authoritative
# ============================================================================

print("Building joined tall LPI table...")


# ============================================================================
# OPTIONAL PROVENANCE FIELD EXPRESSIONS
#
# The actual georeferenced file contains:
#   DBKey.x / DBKey.y
#   ProjectKey.x / ProjectKey.y
#
# Preserve one representative value using COALESCE.
# ============================================================================

DBKEY_SQL = """
COALESCE(
    NULLIF(TRIM("DBKey.x"), ''),
    NULLIF(TRIM("DBKey.y"), '')
) AS DBKey
"""

PROJECTKEY_SQL = """
COALESCE(
    NULLIF(TRIM("ProjectKey.x"), ''),
    NULLIF(TRIM("ProjectKey.y"), '')
) AS ProjectKey
"""


# ============================================================================
# BUILD JOINED TABLE
# ============================================================================

con.execute(f"""
CREATE OR REPLACE TABLE lpi AS

WITH raw AS (

    SELECT
        rid,
        PrimaryKey,

        {DBKEY_SQL},
        {PROJECTKEY_SQL},

        LineKey,
        RecKey,
        layer,
        PointNbr,

        code AS observed_code_raw,

        CASE
            WHEN code IS NULL
                 OR TRIM(code) = ''
            THEN '__NO_CANOPY__'

            ELSE TRIM(code)
        END AS observed_code,

        DateVisited,

        COALESCE(
            TRY_CAST(SampleYear AS INTEGER),
            YEAR(
                TRY_CAST(
                    DateVisited AS TIMESTAMP
                )
            ),
            TRY_CAST(
                SUBSTR(DateVisited, 1, 4)
                AS INTEGER
            )
        ) AS Year,

        TRY_CAST(
            Latitude_NAD83 AS DOUBLE
        ) AS Latitude_NAD83,

        TRY_CAST(
            Longitude_NAD83 AS DOUBLE
        ) AS Longitude_NAD83

    FROM lpi_raw
)

SELECT
    r.*,

    -- ----------------------------------------------------------------------
    -- Explicit dictionary join-presence field.
    --
    -- Do not infer join success from nullable ecological fields.
    -- ----------------------------------------------------------------------

    m.observed_code AS dictionary_match_code,


    -- ----------------------------------------------------------------------
    -- Core interpretation fields from FINAL_MASTER
    -- ----------------------------------------------------------------------

    m.resolved_label,
    m.code_class,
    m.resolution_source,

    m.canonical_code,
    m.canonical_label,

    m.dictionary_resolution_status,
    m.hit_interpretation,


    -- ----------------------------------------------------------------------
    -- USDA taxonomy / ecological traits
    -- ----------------------------------------------------------------------

    m.USDA_accepted_symbol,
    m.USDA_scientific_name,
    m.USDA_common_name,
    m.USDA_family,

    m.USDA_duration,
    m.USDA_growth_habit,
    m.USDA_native_status_L48,
    m.USDA_trait_status,


    -- ----------------------------------------------------------------------
    -- MOSAIC functional groups
    -- ----------------------------------------------------------------------

    m.MOSAIC_FG_structural,
    m.MOSAIC_FG,
    m.FG_resolution_source,


    -- ----------------------------------------------------------------------
    -- Downstream taxon identity
    --
    -- canonical_code is authoritative because it preserves:
    --
    --   POAR2R2 -> PONIN
    --   BAPRV   -> BAPR5
    --   BAPRG   -> BAPR5
    --
    -- Only actual Plant rows contribute to species composition.
    -- ----------------------------------------------------------------------

    CASE
        WHEN m.code_class = 'Plant'
        THEN m.canonical_code
        ELSE NULL
    END AS taxon_id,

    CASE
        WHEN m.code_class = 'Plant'
        THEN m.canonical_label
        ELSE NULL
    END AS taxon_label

FROM raw r

LEFT JOIN species_master m
    ON r.observed_code = m.observed_code
;
""")


# ============================================================================
# JOIN QA
# ============================================================================

join_qa = con.execute("""
SELECT
    COUNT(*) AS n_rows,

    COUNT(
        DISTINCT rid
    ) AS n_unique_rid,

    SUM(
        CASE
            WHEN dictionary_match_code IS NULL
            THEN 1
            ELSE 0
        END
    ) AS n_unmatched_rows,

    COUNT(
        DISTINCT CASE
            WHEN dictionary_match_code IS NULL
            THEN observed_code
        END
    ) AS n_unmatched_codes

FROM lpi
""").df()

display(join_qa)


# ============================================================================
# HARD ACCOUNTING
# ============================================================================

n_joined = con.execute("""
SELECT COUNT(*)
FROM lpi
""").fetchone()[0]

assert n_joined == 16_150_114, (
    f"Joined LPI row count changed: "
    f"{n_joined:,} != 16,150,114"
)


n_unmatched = con.execute("""
SELECT COUNT(*)
FROM lpi
WHERE dictionary_match_code IS NULL
""").fetchone()[0]

assert n_unmatched == 0, (
    f"{n_unmatched:,} raw LPI rows failed to join "
    "to the finalized dictionary."
)


print(
    "PASS: all 16,150,114 LPI rows joined "
    "to the finalized dictionary."
)

Building joined tall LPI table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_unique_rid,n_unmatched_rows,n_unmatched_codes
0,16150114,16150114,442624.0,2


AssertionError: 442,624 raw LPI rows failed to join to the finalized dictionary.

In [17]:
unmatched_codes = con.execute("""
SELECT
    observed_code_raw,
    observed_code,
    COUNT(*) AS n_records,
    COUNT(DISTINCT PrimaryKey) AS n_plot_visits
FROM lpi
WHERE dictionary_match_code IS NULL
GROUP BY
    observed_code_raw,
    observed_code
ORDER BY n_records DESC
""").df()

display(unmatched_codes)

,observed_code_raw,observed_code,n_records,n_plot_visits
0,None,None,442612,8652
1,NA,NA,12,4


In [18]:
# ============================================================================
# DIAGNOSE THE TWO UNMATCHED CODE REPRESENTATIONS
# ============================================================================

print("RAW LPI representation:")
display(
    con.execute("""
        SELECT
            code,
            COUNT(*) AS n_rows,
            typeof(code) AS duckdb_type
        FROM lpi_raw
        WHERE code IS NULL
           OR TRIM(code) = ''
           OR UPPER(TRIM(code)) = 'NA'
           OR UPPER(TRIM(code)) = '__NO_CANOPY__'
        GROUP BY
            code,
            typeof(code)
        ORDER BY n_rows DESC
    """).df()
)

print("\nRepresentation after raw transformation in lpi:")
display(
    con.execute("""
        SELECT
            observed_code_raw,
            observed_code,
            dictionary_match_code,
            COUNT(*) AS n_rows
        FROM lpi
        WHERE dictionary_match_code IS NULL
        GROUP BY
            observed_code_raw,
            observed_code,
            dictionary_match_code
        ORDER BY n_rows DESC
    """).df()
)

print("\nRelevant dictionary rows:")
display(
    con.execute("""
        SELECT
            observed_code,
            code,
            canonical_code,
            code_class,
            dictionary_resolution_status,
            hit_interpretation,
            n_records
        FROM species_master
        WHERE observed_code IS NULL
           OR TRIM(observed_code) = ''
           OR UPPER(TRIM(observed_code)) IN (
               'NA',
               '__NO_CANOPY__'
           )
           OR UPPER(TRIM(code)) IN (
               'NA',
               '__NO_CANOPY__'
           )
        ORDER BY n_records DESC
    """).df()
)

print("\nDictionary NULL accounting:")
display(
    con.execute("""
        SELECT
            COUNT(*) AS dictionary_rows,
            SUM(CASE WHEN observed_code IS NULL THEN 1 ELSE 0 END)
                AS null_observed_code,
            SUM(CASE WHEN code IS NULL THEN 1 ELSE 0 END)
                AS null_code
        FROM species_master
    """).df()
)

RAW LPI representation:


,code,n_rows,duckdb_type
0,NA,12,VARCHAR
1,,1,VARCHAR



Representation after raw transformation in lpi:


,observed_code_raw,observed_code,dictionary_match_code,n_rows
0,None,None,None,442612
1,NA,NA,None,12



Relevant dictionary rows:


,observed_code,code,canonical_code,code_class,dictionary_resolution_status,hit_interpretation,n_records
0,__NO_CANOPY__,None,__NO_CANOPY__,NoCanopy,NoCanopy,NO_CANOPY,442612



Dictionary NULL accounting:


,dictionary_rows,null_observed_code,null_code
0,10960,0.0,1.0


In [20]:
# ============================================================================
# INSPECT ALL SURVEY CONTEXT FOR LITERAL "NA" CODES
#
# READ-ONLY
# ============================================================================

na_hits = con.execute("""
SELECT
    rid,
    PrimaryKey,

    COALESCE(
        NULLIF(TRIM("DBKey.x"), ''),
        NULLIF(TRIM("DBKey.y"), '')
    ) AS DBKey,

    COALESCE(
        NULLIF(TRIM("ProjectKey.x"), ''),
        NULLIF(TRIM("ProjectKey.y"), '')
    ) AS ProjectKey,

    PlotID,
    SampleYear,
    DateVisited,

    LineKey,
    PointNbr,
    layer,
    code,

    "source.x",
    "source.y",

    Latitude_NAD83,
    Longitude_NAD83,
    EcologicalSiteID

FROM lpi_raw

WHERE UPPER(TRIM(code)) = 'NA'

ORDER BY
    PrimaryKey,
    LineKey,
    TRY_CAST(PointNbr AS INTEGER),
    layer
""").df()

display(na_hits)
# ============================================================================
# SHOW COMPLETE PIN / VISIT CONTEXT AROUND EVERY "NA" HIT
# ============================================================================

na_context = con.execute("""
WITH na_pins AS (
    SELECT DISTINCT
        PrimaryKey,
        LineKey,
        PointNbr
    FROM lpi_raw
    WHERE UPPER(TRIM(code)) = 'NA'
)

SELECT
    r.PrimaryKey,
    r.PlotID,
    r.SampleYear,
    r.DateVisited,

    r.LineKey,
    r.PointNbr,
    r.layer,
    r.code,

    r."ProjectKey.x",
    r."ProjectKey.y",
    r."source.x",
    r."source.y",

    r.rid

FROM lpi_raw r

INNER JOIN na_pins n
    ON r.PrimaryKey = n.PrimaryKey
   AND r.LineKey = n.LineKey
   AND r.PointNbr = n.PointNbr

ORDER BY
    r.PrimaryKey,
    r.LineKey,
    TRY_CAST(r.PointNbr AS INTEGER),

    CASE r.layer
        WHEN 'TopCanopy'   THEN 1
        WHEN 'Lower1'      THEN 2
        WHEN 'Lower2'      THEN 3
        WHEN 'Lower3'      THEN 4
        WHEN 'Lower4'      THEN 5
        WHEN 'Lower5'      THEN 6
        WHEN 'Lower6'      THEN 7
        WHEN 'Lower7'      THEN 8
        WHEN 'SoilSurface' THEN 9
        ELSE 99
    END
""").df()

display(na_context)

,rid,PrimaryKey,DBKey,ProjectKey,PlotID,SampleYear,DateVisited,LineKey,PointNbr,layer,code,source.x,source.y,Latitude_NAD83,Longitude_NAD83,EcologicalSiteID
0,68799188,20194145384011B3,BLM_Natl_AIM_LMF_Public.gdb,BLM_AIM,20194145384011B3,2019,2019-06-01T00:00:00.000Z,nesw,30,TopCanopy,NA,LMF,LMF,42.292595,-118.0004683,023XY318OR
1,70707787,20204145374333B2,BLM_Natl_AIM_LMF_Public.gdb,BLM_AIM,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nesw,54,TopCanopy,NA,LMF,LMF,42.313615,-117.6851633,025XY014OR
2,72458041,20204145374333B2,BLM_Natl_AIM_LMF_Public.gdb,BLM_AIM,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nesw,87,TopCanopy,NA,LMF,LMF,42.313615,-117.6851633,025XY014OR
3,71130481,20204145374333B2,BLM_Natl_AIM_LMF_Public.gdb,BLM_AIM,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nwse,6,TopCanopy,NA,LMF,LMF,42.313615,-117.6851633,025XY014OR
4,70885801,20204145374333B2,BLM_Natl_AIM_LMF_Public.gdb,BLM_AIM,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nwse,36,TopCanopy,NA,LMF,LMF,42.313615,-117.6851633,025XY014OR
5,69762841,20234145384008B3,BLM_Natl_AIM_LMF_Public.gdb,BLM_AIM,20234145384008B3,2023,2023-07-16T00:00:00.000Z,nesw,117,TopCanopy,NA,LMF,LMF,42.292563,-118.0574408,R023XY302OR
6,71289580,20234145384008B3,BLM_Natl_AIM_LMF_Public.gdb,BLM_AIM,20234145384008B3,2023,2023-07-16T00:00:00.000Z,nwse,9,Lower1,NA,LMF,LMF,42.292563,-118.0574408,R023XY302OR
7,72595529,20234145384008B3,BLM_Natl_AIM_LMF_Public.gdb,BLM_AIM,20234145384008B3,2023,2023-07-16T00:00:00.000Z,nwse,33,TopCanopy,NA,LMF,LMF,42.292563,-118.0574408,R023XY302OR
8,72440264,20234145384008B3,BLM_Natl_AIM_LMF_Public.gdb,BLM_AIM,20234145384008B3,2023,2023-07-16T00:00:00.000Z,nwse,39,TopCanopy,NA,LMF,LMF,42.292563,-118.0574408,R023XY302OR
9,71789158,20234145384008B3,BLM_Natl_AIM_LMF_Public.gdb,BLM_AIM,20234145384008B3,2023,2023-07-16T00:00:00.000Z,nwse,87,TopCanopy,NA,LMF,LMF,42.292563,-118.0574408,R023XY302OR


,PrimaryKey,PlotID,SampleYear,DateVisited,LineKey,PointNbr,layer,code,ProjectKey.x,ProjectKey.y,source.x,source.y,rid
0,20194145384011B3,20194145384011B3,2019,2019-06-01T00:00:00.000Z,nesw,30,TopCanopy,NA,BLM_AIM,BLM_AIM,LMF,LMF,68799188
1,20194145384011B3,20194145384011B3,2019,2019-06-01T00:00:00.000Z,nesw,30,Lower1,ELEL5,BLM_AIM,BLM_AIM,LMF,LMF,72070532
2,20194145384011B3,20194145384011B3,2019,2019-06-01T00:00:00.000Z,nesw,30,SoilSurface,ELEL5,BLM_AIM,BLM_AIM,LMF,LMF,70884286
3,20204145374333B2,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nesw,54,TopCanopy,NA,BLM_AIM,BLM_AIM,LMF,LMF,70707787
4,20204145374333B2,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nesw,54,Lower1,POSE,BLM_AIM,BLM_AIM,LMF,LMF,72433412
5,20204145374333B2,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nesw,54,SoilSurface,POSE,BLM_AIM,BLM_AIM,LMF,LMF,72530143
6,20204145374333B2,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nesw,87,TopCanopy,NA,BLM_AIM,BLM_AIM,LMF,LMF,72458041
7,20204145374333B2,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nesw,87,Lower1,POSE,BLM_AIM,BLM_AIM,LMF,LMF,70434679
8,20204145374333B2,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nesw,87,SoilSurface,S,BLM_AIM,BLM_AIM,LMF,LMF,68506632
9,20204145374333B2,20204145374333B2,2020,2020-05-22T00:00:00.000Z,nwse,6,TopCanopy,NA,BLM_AIM,BLM_AIM,LMF,LMF,71130481


In [19]:
# ============================================================================
# 6. DEFINE VALID VISITS AND UNKNOWN-CODE QA
#
# Processing universe:
#   1. Identify every PrimaryKey × Year containing PlantBase.
#   2. Exclude that entire visit from all composition calculations.
#   3. For retained visits, calculate UnknownCode percentage across ALL
#      protocol LPI hit rows (TopCanopy, Lower1-Lower7, SoilSurface).
#
# UnknownCode is a QA FLAG, not an exclusion criterion.
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE excluded_plantbase_visits AS
SELECT DISTINCT
    PrimaryKey,
    Year
FROM lpi
WHERE code_class = 'PlantBase'
  AND PrimaryKey IS NOT NULL
  AND Year IS NOT NULL
;
""")

plantbase_exclusion_qa = con.execute("""
SELECT
    Year,
    COUNT(*) AS n_excluded_plot_visits
FROM excluded_plantbase_visits
GROUP BY Year
ORDER BY Year
""").df()

print("PlantBase visit exclusions by year:")
display(plantbase_exclusion_qa)

plantbase_exclusion_qa.to_csv(
    QA_DIR / "LPI_PlantBase_excluded_visits_by_year.csv",
    index=False,
)

con.execute("""
CREATE OR REPLACE TABLE lpi_valid AS
SELECT l.*
FROM lpi l
LEFT JOIN excluded_plantbase_visits e
    ON l.PrimaryKey = e.PrimaryKey
   AND l.Year = e.Year
WHERE e.PrimaryKey IS NULL
;
""")

visit_universe_qa = con.execute("""
SELECT
    (SELECT COUNT(DISTINCT (PrimaryKey, Year)) FROM lpi) AS n_all_visits,
    (SELECT COUNT(*) FROM excluded_plantbase_visits) AS n_plantbase_excluded,
    (SELECT COUNT(DISTINCT (PrimaryKey, Year)) FROM lpi_valid) AS n_retained_visits,
    (SELECT COUNT(*) FROM lpi) AS n_all_rows,
    (SELECT COUNT(*) FROM lpi_valid) AS n_retained_rows
""").df()

display(visit_universe_qa)

con.execute(f"""
CREATE OR REPLACE TABLE visit_unknown_code_qa AS
SELECT
    PrimaryKey,
    Year,

    COUNT(*) AS n_all_lpi_hits,

    SUM(
        CASE
            WHEN dictionary_resolution_status = 'UnknownCode'
            THEN 1
            ELSE 0
        END
    ) AS n_unknown_code_hits,

    100.0 * SUM(
        CASE
            WHEN dictionary_resolution_status = 'UnknownCode'
            THEN 1
            ELSE 0
        END
    ) / COUNT(*) AS unknown_code_pct_all_hits,

    (
        100.0 * SUM(
            CASE
                WHEN dictionary_resolution_status = 'UnknownCode'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*)
    ) > {UNKNOWN_CODE_FLAG_THRESHOLD_PCT} AS unknown_code_gt5pct

FROM lpi_valid
WHERE layer IN (
    'TopCanopy',
    'Lower1', 'Lower2', 'Lower3', 'Lower4',
    'Lower5', 'Lower6', 'Lower7',
    'SoilSurface'
)
GROUP BY PrimaryKey, Year
;
""")

unknown_flag_summary = con.execute("""
SELECT
    Year,
    COUNT(*) AS n_retained_plot_visits,
    SUM(CASE WHEN unknown_code_gt5pct THEN 1 ELSE 0 END) AS n_flagged_gt5pct,
    MAX(unknown_code_pct_all_hits) AS max_unknown_code_pct,
    AVG(unknown_code_pct_all_hits) AS mean_unknown_code_pct
FROM visit_unknown_code_qa
GROUP BY Year
ORDER BY Year
""").df()

print("UnknownCode visit-level QA:")
display(unknown_flag_summary)

unknown_flag_summary.to_csv(
    QA_DIR / "LPI_UnknownCode_flag_summary_by_year.csv",
    index=False,
)

unknown_flagged_visits = con.execute("""
SELECT *
FROM visit_unknown_code_qa
WHERE unknown_code_gt5pct
ORDER BY Year, unknown_code_pct_all_hits DESC, PrimaryKey
""").df()

unknown_flagged_visits.to_csv(
    QA_DIR / "LPI_UnknownCode_gt5pct_flagged_visits.csv",
    index=False,
)

print(
    "Visits flagged >5% UnknownCode:",
    f"{len(unknown_flagged_visits):,}"
)


PlantBase visit exclusions by year:


,Year,n_excluded_plot_visits
0,2018,257
1,2019,155
2,2020,71
3,2021,64
4,2022,146
5,2023,141
6,2024,271


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_all_visits,n_plantbase_excluded,n_retained_visits,n_all_rows,n_retained_rows
0,51812,1105,50707,16150114,15905451


UnknownCode visit-level QA:


,Year,n_retained_plot_visits,n_flagged_gt5pct,max_unknown_code_pct,mean_unknown_code_pct
0,2018,7056,0.0,2.985075,0.004402
1,2019,8884,1.0,7.476636,0.005168
2,2020,7274,1.0,5.314010,0.005369
3,2021,8533,0.0,3.010753,0.004352
4,2022,9185,0.0,3.225806,0.002457
5,2023,8349,1.0,9.686610,0.006279
6,2024,1418,0.0,0.235849,0.000166
7,2025,8,0.0,0.000000,0.000000


Visits flagged >5% UnknownCode: 3


In [21]:
# ============================================================================
# 6. DEFINE VALID VISITS AND UNKNOWN-CODE QA
#
# VISIT EXCLUSIONS
# ----------------
# 1. PlantBase:
#       Exclude the entire PrimaryKey × Year visit.
#
# 2. Literal "NA":
#       Only 12 records across 4 visits.
#       Protocol meaning is ambiguous, so exclude the entire affected visit.
#
# 3. UnknownCode threshold:
#       Evaluate uncertainty at the TopCanopy / first-hit level.
#
#       denominator =
#           all unique pin drops in the visit
#
#       numerator =
#           unique pin drops whose TopCanopy record is UnknownCode
#
#       Exclude entire visit when unknown_topcanopy_pct > 1%.
#
#
# NULL CODE HANDLING
# ------------------
# Raw code = NULL/None is mapped to the established __NO_CANOPY__
# dictionary record. This is NOT treated as UnknownCode.
#
#
# IMPORTANT DENOMINATOR PRINCIPLE
# -------------------------------
# A pin drop is the sampling trial:
#
#     PrimaryKey × Year × LineKey × PointNbr
#
# Both first-hit and multilayer composition use the number of unique
# pin drops as their denominator.
#
# Lower canopy layers do not create additional trials.
# ============================================================================


# ============================================================================
# A. NORMALIZE RAW NULL CODES TO THE ESTABLISHED __NO_CANOPY__ RECORD
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE lpi_normalized AS

SELECT

    l.* EXCLUDE (
        observed_code,
        dictionary_match_code,

        resolved_label,
        code_class,
        resolution_source,

        canonical_code,
        canonical_label,

        dictionary_resolution_status,
        hit_interpretation,

        USDA_accepted_symbol,
        USDA_scientific_name,
        USDA_common_name,
        USDA_family,

        USDA_duration,
        USDA_growth_habit,
        USDA_native_status_L48,
        USDA_trait_status,

        MOSAIC_FG_structural,
        MOSAIC_FG,
        FG_resolution_source,

        taxon_id,
        taxon_label
    ),

    -- ----------------------------------------------------------------------
    -- Effective observed code
    -- ----------------------------------------------------------------------

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN '__NO_CANOPY__'
        ELSE TRIM(l.observed_code_raw)
    END AS observed_code,


    -- ----------------------------------------------------------------------
    -- Dictionary join marker
    -- ----------------------------------------------------------------------

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.observed_code
        ELSE l.dictionary_match_code
    END AS dictionary_match_code,


    -- ----------------------------------------------------------------------
    -- Dictionary interpretation fields
    -- ----------------------------------------------------------------------

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.resolved_label
        ELSE l.resolved_label
    END AS resolved_label,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.code_class
        ELSE l.code_class
    END AS code_class,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.resolution_source
        ELSE l.resolution_source
    END AS resolution_source,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.canonical_code
        ELSE l.canonical_code
    END AS canonical_code,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.canonical_label
        ELSE l.canonical_label
    END AS canonical_label,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.dictionary_resolution_status
        ELSE l.dictionary_resolution_status
    END AS dictionary_resolution_status,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.hit_interpretation
        ELSE l.hit_interpretation
    END AS hit_interpretation,


    -- ----------------------------------------------------------------------
    -- USDA fields
    -- ----------------------------------------------------------------------

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.USDA_accepted_symbol
        ELSE l.USDA_accepted_symbol
    END AS USDA_accepted_symbol,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.USDA_scientific_name
        ELSE l.USDA_scientific_name
    END AS USDA_scientific_name,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.USDA_common_name
        ELSE l.USDA_common_name
    END AS USDA_common_name,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.USDA_family
        ELSE l.USDA_family
    END AS USDA_family,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.USDA_duration
        ELSE l.USDA_duration
    END AS USDA_duration,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.USDA_growth_habit
        ELSE l.USDA_growth_habit
    END AS USDA_growth_habit,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.USDA_native_status_L48
        ELSE l.USDA_native_status_L48
    END AS USDA_native_status_L48,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.USDA_trait_status
        ELSE l.USDA_trait_status
    END AS USDA_trait_status,


    -- ----------------------------------------------------------------------
    -- Functional groups
    -- ----------------------------------------------------------------------

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.MOSAIC_FG_structural
        ELSE l.MOSAIC_FG_structural
    END AS MOSAIC_FG_structural,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.MOSAIC_FG
        ELSE l.MOSAIC_FG
    END AS MOSAIC_FG,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN nc.FG_resolution_source
        ELSE l.FG_resolution_source
    END AS FG_resolution_source,


    -- ----------------------------------------------------------------------
    -- Taxon identity
    --
    -- NoCanopy is not a taxon.
    -- ----------------------------------------------------------------------

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN NULL
        ELSE l.taxon_id
    END AS taxon_id,

    CASE
        WHEN l.observed_code_raw IS NULL
        THEN NULL
        ELSE l.taxon_label
    END AS taxon_label


FROM lpi l

LEFT JOIN species_master nc
    ON nc.observed_code = '__NO_CANOPY__'
;
""")


# ============================================================================
# B. VERIFY NULL -> __NO_CANOPY__
# ============================================================================

null_code_qa = con.execute("""
SELECT
    observed_code,
    code_class,
    dictionary_resolution_status,
    hit_interpretation,
    COUNT(*) AS n_rows

FROM lpi_normalized

WHERE observed_code_raw IS NULL

GROUP BY
    observed_code,
    code_class,
    dictionary_resolution_status,
    hit_interpretation
;
""").df()

print("NULL-code normalization:")
display(null_code_qa)


n_bad_nulls = con.execute("""
SELECT COUNT(*)

FROM lpi_normalized

WHERE observed_code_raw IS NULL
  AND (
        observed_code <> '__NO_CANOPY__'
        OR dictionary_match_code IS NULL
      )
;
""").fetchone()[0]

assert n_bad_nulls == 0, (
    f"{n_bad_nulls:,} NULL-code rows failed "
    "to resolve to __NO_CANOPY__."
)


# ============================================================================
# C. EXCLUDE VISITS CONTAINING LITERAL "NA"
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE excluded_na_visits AS

SELECT DISTINCT
    PrimaryKey,
    Year

FROM lpi_normalized

WHERE UPPER(TRIM(observed_code_raw)) = 'NA'
  AND PrimaryKey IS NOT NULL
  AND Year IS NOT NULL
;
""")


na_exclusion_qa = con.execute("""
SELECT
    Year,
    COUNT(*) AS n_excluded_plot_visits

FROM excluded_na_visits

GROUP BY Year
ORDER BY Year
;
""").df()

print("\nLiteral-NA visit exclusions:")
display(na_exclusion_qa)


n_na_hits = con.execute("""
SELECT COUNT(*)

FROM lpi_normalized

WHERE UPPER(TRIM(observed_code_raw)) = 'NA'
;
""").fetchone()[0]

n_na_visits = con.execute("""
SELECT COUNT(*)
FROM excluded_na_visits
;
""").fetchone()[0]

print(f"Literal NA hits: {n_na_hits:,}")
print(f"NA-affected visits: {n_na_visits:,}")

assert n_na_hits == 12, (
    f"Expected 12 literal NA records; found {n_na_hits:,}."
)


# ============================================================================
# D. PRELIMINARY RETAINED TABLE
#
# PlantBase + literal-NA visits removed.
# Unknown threshold not yet applied.
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE lpi_pre_unknown_filter AS

SELECT l.*

FROM lpi_normalized l

LEFT JOIN excluded_plantbase_visits pb
    ON l.PrimaryKey = pb.PrimaryKey
   AND l.Year = pb.Year

LEFT JOIN excluded_na_visits na
    ON l.PrimaryKey = na.PrimaryKey
   AND l.Year = na.Year

WHERE pb.PrimaryKey IS NULL
  AND na.PrimaryKey IS NULL
;
""")


# ============================================================================
# E. UNIQUE PIN DROPS FOR UNKNOWN-CODE QA
#
# This is the denominator concept used throughout the notebook.
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE qa_pins AS

SELECT DISTINCT
    PrimaryKey,
    Year,
    LineKey,
    PointNbr

FROM lpi_pre_unknown_filter

WHERE PrimaryKey IS NOT NULL
  AND Year IS NOT NULL
  AND LineKey IS NOT NULL
  AND PointNbr IS NOT NULL
;
""")


# ============================================================================
# F. IDENTIFY PINS WITH UNKNOWN TOP-CANOPY CODES
#
# One pin counts at most once toward the unknown numerator.
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE qa_unknown_topcanopy_pins AS

SELECT DISTINCT
    PrimaryKey,
    Year,
    LineKey,
    PointNbr

FROM lpi_pre_unknown_filter

WHERE layer = 'TopCanopy'

  AND dictionary_resolution_status = 'UnknownCode'

  AND PrimaryKey IS NOT NULL
  AND Year IS NOT NULL
  AND LineKey IS NOT NULL
  AND PointNbr IS NOT NULL
;
""")


# ============================================================================
# G. CALCULATE TOP-CANOPY UNKNOWN PERCENTAGE
#
# unknown_topcanopy_pct =
#
#     unknown TopCanopy pins
#     ---------------------- × 100
#     all sampled pins
#
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE topcanopy_unknown_qa AS

WITH totals AS (

    SELECT
        PrimaryKey,
        Year,
        COUNT(*) AS n_pins

    FROM qa_pins

    GROUP BY
        PrimaryKey,
        Year
),

unknowns AS (

    SELECT
        PrimaryKey,
        Year,
        COUNT(*) AS n_unknown_topcanopy_pins

    FROM qa_unknown_topcanopy_pins

    GROUP BY
        PrimaryKey,
        Year
)

SELECT
    t.PrimaryKey,
    t.Year,

    t.n_pins,

    COALESCE(
        u.n_unknown_topcanopy_pins,
        0
    ) AS n_unknown_topcanopy_pins,

    100.0
        * COALESCE(
            u.n_unknown_topcanopy_pins,
            0
          )
        / t.n_pins
        AS unknown_topcanopy_pct,

    CASE
        WHEN
            100.0
            * COALESCE(
                u.n_unknown_topcanopy_pins,
                0
              )
            / t.n_pins
            > 1.0

        THEN TRUE
        ELSE FALSE
    END AS unknown_topcanopy_gt1pct

FROM totals t

LEFT JOIN unknowns u
    ON t.PrimaryKey = u.PrimaryKey
   AND t.Year = u.Year
;
""")


# ============================================================================
# H. VISITS EXCLUDED FOR >1% UNKNOWN TOP-CANOPY PINS
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE excluded_unknown_visits AS

SELECT
    PrimaryKey,
    Year

FROM topcanopy_unknown_qa

WHERE unknown_topcanopy_gt1pct = TRUE
;
""")


unknown_year_qa = con.execute("""
SELECT
    Year,

    COUNT(*) AS n_pre_threshold_visits,

    SUM(
        CASE
            WHEN unknown_topcanopy_gt1pct
            THEN 1
            ELSE 0
        END
    ) AS n_excluded_gt1pct,

    MAX(unknown_topcanopy_pct)
        AS max_unknown_topcanopy_pct,

    AVG(unknown_topcanopy_pct)
        AS mean_unknown_topcanopy_pct

FROM topcanopy_unknown_qa

GROUP BY Year
ORDER BY Year
;
""").df()

print("\nTop-canopy UnknownCode QA:")
display(unknown_year_qa)


# ============================================================================
# I. FINAL RETAINED TALL TABLE
#
# Cell 7 uses this table.
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE lpi_valid AS

SELECT l.*

FROM lpi_pre_unknown_filter l

LEFT JOIN excluded_unknown_visits u
    ON l.PrimaryKey = u.PrimaryKey
   AND l.Year = u.Year

WHERE u.PrimaryKey IS NULL
;
""")


# ============================================================================
# J. FINAL ACCOUNTING
# ============================================================================

final_visit_qa = con.execute("""
SELECT

    (
        SELECT COUNT(DISTINCT (PrimaryKey, Year))
        FROM lpi_normalized
        WHERE PrimaryKey IS NOT NULL
          AND Year IS NOT NULL
    ) AS n_all_visits,

    (
        SELECT COUNT(*)
        FROM excluded_plantbase_visits
    ) AS n_plantbase_excluded,

    (
        SELECT COUNT(*)
        FROM excluded_na_visits
    ) AS n_na_excluded,

    (
        SELECT COUNT(*)
        FROM excluded_unknown_visits
    ) AS n_unknown_gt1pct_excluded,

    (
        SELECT COUNT(DISTINCT (PrimaryKey, Year))
        FROM lpi_valid
        WHERE PrimaryKey IS NOT NULL
          AND Year IS NOT NULL
    ) AS n_retained_visits,

    (
        SELECT COUNT(*)
        FROM lpi_valid
    ) AS n_retained_rows
;
""").df()

print("\nFinal retained-visit accounting:")
display(final_visit_qa)


# ============================================================================
# K. HARD GUARDS
# ============================================================================

n_na_remaining = con.execute("""
SELECT COUNT(*)

FROM lpi_valid

WHERE UPPER(TRIM(observed_code_raw)) = 'NA'
;
""").fetchone()[0]

assert n_na_remaining == 0, (
    f"{n_na_remaining:,} literal NA rows remain in lpi_valid."
)


n_threshold_failures = con.execute("""
SELECT COUNT(*)

FROM topcanopy_unknown_qa q

INNER JOIN (
    SELECT DISTINCT
        PrimaryKey,
        Year
    FROM lpi_valid
) v
    ON q.PrimaryKey = v.PrimaryKey
   AND q.Year = v.Year

WHERE q.unknown_topcanopy_pct > 1.0
;
""").fetchone()[0]

assert n_threshold_failures == 0, (
    f"{n_threshold_failures:,} visits above the 1% "
    "TopCanopy UnknownCode threshold remain."
)


print()
print("PASS: lpi_valid is ready for Cell 7.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

NULL-code normalization:


,observed_code,code_class,dictionary_resolution_status,hit_interpretation,n_rows



Literal-NA visit exclusions:


,Year,n_excluded_plot_visits
0,2019,1
1,2020,1
2,2023,2


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Final retained-visit accounting:


,n_all_visits,n_plantbase_excluded,n_na_excluded,n_unknown_gt1pct_excluded,n_retained_visits,n_retained_rows
0,51812,1105,4,77,50626,15875184



PASS: lpi_valid is ready for Cell 7.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique pin-drop denominator QA:


,n_plot_years,min_points,median_points,max_points,mean_points
0,50626,6,150.0,1200,135.089381



Pin-drop denominator by year:


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Largest line-level pin counts:


,PrimaryKey,Year,LineKey,n_pin_drops
0,15120315082630972018-06-26,2018,1512031508283479,400
1,17062316455172542019-12-10,2019,1706231645524497,400
2,17062316455172542019-12-10,2019,1706231645526272,400
3,17062316455172542019-12-10,2019,1706231645528128,400
4,15120315082630972019-05-15,2019,1512031508289527,399
5,15120315082630972018-06-26,2018,1512031508281623,398
6,15120315082630972018-06-26,2018,1512031508289527,398
7,19020816041349872018-12-21,2018,190208160414117,398
8,15120315082630972019-05-15,2019,1512031508281623,398
9,19020816041349872020-04-06,2020,190208160414117,398


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


PASS: common pin-drop denominator is established.
Both first-hit and multilayer composition will use plot_totals.n_points as the denominator.


we need to troubleshoot these odd n_pin_drops 

In [23]:
# ============================================================================
# DIAGNOSE SUSPICIOUSLY LARGE PIN COUNTS
#
# READ ONLY
#
# Examine whether high n_pin_drops values are explained by:
#   - line length
#   - sampling interval
#   - direction
#   - protocol / source
#   - multiple forms / records
#   - PointNbr range and uniqueness
#
# No denominator logic is changed here.
# ============================================================================


# ============================================================================
# A. IDENTIFY THE 50 LARGEST APPARENT LINES
# ============================================================================

con.execute("""
CREATE OR REPLACE TEMP TABLE suspicious_lines AS

SELECT
    PrimaryKey,
    Year,
    LineKey,
    COUNT(*) AS n_pin_drops

FROM pins

GROUP BY
    PrimaryKey,
    Year,
    LineKey

ORDER BY n_pin_drops DESC

LIMIT 50
;
""")


print("Largest apparent lines:")
display(
    con.execute("""
        SELECT *
        FROM suspicious_lines
        ORDER BY n_pin_drops DESC
    """).df()
)


# ============================================================================
# B. PULL PROTOCOL METADATA FOR THOSE LINES
#
# Use lpi_raw because it contains the original survey metadata.
# ============================================================================

large_line_metadata = con.execute("""
SELECT

    r.PrimaryKey,
    TRY_CAST(r.SampleYear AS INTEGER) AS Year,
    r.LineKey,

    s.n_pin_drops,

    -- ----------------------------------------------------------------------
    -- provenance / protocol
    -- ----------------------------------------------------------------------

    r.PlotID,

    COALESCE(
        NULLIF(TRIM(r."ProjectKey.x"), ''),
        NULLIF(TRIM(r."ProjectKey.y"), '')
    ) AS ProjectKey,

    COALESCE(
        NULLIF(TRIM(r."DBKey.x"), ''),
        NULLIF(TRIM(r."DBKey.y"), '')
    ) AS DBKey,

    COALESCE(
        NULLIF(TRIM(r."source.x"), ''),
        NULLIF(TRIM(r."source.y"), '')
    ) AS source,

    r.FormType,
    r.Direction,

    r.Measure,
    r.LineLengthAmount,
    r.SpacingIntervalAmount,
    r.SpacingType,

    -- ----------------------------------------------------------------------
    -- PointNbr structure
    -- ----------------------------------------------------------------------

    COUNT(DISTINCT r.PointNbr)
        AS n_distinct_pointnbr,

    MIN(
        TRY_CAST(r.PointNbr AS DOUBLE)
    ) AS min_pointnbr,

    MAX(
        TRY_CAST(r.PointNbr AS DOUBLE)
    ) AS max_pointnbr,

    -- ----------------------------------------------------------------------
    -- raw record structure
    -- ----------------------------------------------------------------------

    COUNT(*) AS n_raw_layer_records,

    COUNT(DISTINCT r.layer)
        AS n_distinct_layers,

    STRING_AGG(
        DISTINCT r.layer,
        ', '
        ORDER BY r.layer
    ) AS layers_present

FROM lpi_raw r

INNER JOIN suspicious_lines s
    ON r.PrimaryKey = s.PrimaryKey
   AND TRY_CAST(r.SampleYear AS INTEGER) = s.Year
   AND r.LineKey = s.LineKey

GROUP BY
    r.PrimaryKey,
    TRY_CAST(r.SampleYear AS INTEGER),
    r.LineKey,
    s.n_pin_drops,

    r.PlotID,

    COALESCE(
        NULLIF(TRIM(r."ProjectKey.x"), ''),
        NULLIF(TRIM(r."ProjectKey.y"), '')
    ),

    COALESCE(
        NULLIF(TRIM(r."DBKey.x"), ''),
        NULLIF(TRIM(r."DBKey.y"), '')
    ),

    COALESCE(
        NULLIF(TRIM(r."source.x"), ''),
        NULLIF(TRIM(r."source.y"), '')
    ),

    r.FormType,
    r.Direction,

    r.Measure,
    r.LineLengthAmount,
    r.SpacingIntervalAmount,
    r.SpacingType

ORDER BY
    s.n_pin_drops DESC,
    r.PrimaryKey,
    r.LineKey
;
""").df()


print("\nProtocol metadata for largest apparent lines:")
display(large_line_metadata)


# ============================================================================
# C. INSPECT PointNbr VALUES FOR THE 10 LARGEST LINES
#
# This will tell us whether a 400-pin line actually has something like:
#
#     1, 2, 3 ... 400
#
# or whether PointNbr has another structure.
# ============================================================================

largest_10_point_structure = con.execute("""
WITH top10 AS (

    SELECT *
    FROM suspicious_lines

    ORDER BY n_pin_drops DESC

    LIMIT 10
)

SELECT

    p.PrimaryKey,
    p.Year,
    p.LineKey,

    COUNT(*) AS n_distinct_points,

    MIN(
        TRY_CAST(p.PointNbr AS DOUBLE)
    ) AS min_pointnbr,

    MAX(
        TRY_CAST(p.PointNbr AS DOUBLE)
    ) AS max_pointnbr,

    COUNT(
        DISTINCT TRY_CAST(p.PointNbr AS DOUBLE)
    ) AS n_numeric_pointnbr,

    STRING_AGG(
        p.PointNbr,
        ', '
        ORDER BY TRY_CAST(p.PointNbr AS DOUBLE)
    ) AS point_numbers

FROM pins p

INNER JOIN top10 t
    ON p.PrimaryKey = t.PrimaryKey
   AND p.Year = t.Year
   AND p.LineKey = t.LineKey

GROUP BY
    p.PrimaryKey,
    p.Year,
    p.LineKey

ORDER BY
    n_distinct_points DESC
;
""").df()


print("\nPointNbr structure for 10 largest lines:")
display(largest_10_point_structure)


# ============================================================================
# D. SUMMARIZE LARGE-LINE COUNTS BY SOURCE / PROTOCOL
#
# This helps determine whether the phenomenon is concentrated in one
# contributing database.
# ============================================================================

large_line_source_summary = con.execute("""
WITH line_counts AS (

    SELECT
        p.PrimaryKey,
        p.Year,
        p.LineKey,
        COUNT(*) AS n_pin_drops

    FROM pins p

    GROUP BY
        p.PrimaryKey,
        p.Year,
        p.LineKey
),

line_metadata AS (

    SELECT DISTINCT

        r.PrimaryKey,
        TRY_CAST(r.SampleYear AS INTEGER) AS Year,
        r.LineKey,

        COALESCE(
            NULLIF(TRIM(r."source.x"), ''),
            NULLIF(TRIM(r."source.y"), '')
        ) AS source,

        COALESCE(
            NULLIF(TRIM(r."DBKey.x"), ''),
            NULLIF(TRIM(r."DBKey.y"), '')
        ) AS DBKey,

        r.FormType,
        r.LineLengthAmount,
        r.SpacingIntervalAmount,
        r.SpacingType

    FROM lpi_raw r
)

SELECT

    m.source,
    m.DBKey,
    m.FormType,
    m.LineLengthAmount,
    m.SpacingIntervalAmount,
    m.SpacingType,

    COUNT(*) AS n_lines,

    MIN(c.n_pin_drops) AS min_pin_drops,
    MEDIAN(c.n_pin_drops) AS median_pin_drops,
    MAX(c.n_pin_drops) AS max_pin_drops

FROM line_counts c

LEFT JOIN line_metadata m
    ON c.PrimaryKey = m.PrimaryKey
   AND c.Year = m.Year
   AND c.LineKey = m.LineKey

WHERE c.n_pin_drops > 200

GROUP BY
    m.source,
    m.DBKey,
    m.FormType,
    m.LineLengthAmount,
    m.SpacingIntervalAmount,
    m.SpacingType

ORDER BY
    max_pin_drops DESC,
    n_lines DESC
;
""").df()


print("\nLines >200 apparent pin drops, summarized by protocol:")
display(large_line_source_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Largest apparent lines:


,PrimaryKey,Year,LineKey,n_pin_drops
0,17062316455172542019-12-10,2019,1706231645526272,400
1,17062316455172542019-12-10,2019,1706231645524497,400
2,15120315082630972018-06-26,2018,1512031508283479,400
3,17062316455172542019-12-10,2019,1706231645528128,400
4,15120315082630972019-05-15,2019,1512031508289527,399
5,19020816041349872021-03-29,2021,190208160414117,398
6,19020816041349872022-02-15,2022,190208160414117,398
7,15120315082630972019-05-15,2019,1512031508281623,398
8,19020816041349872020-10-19,2020,190208160414117,398
9,15120315082630972018-06-26,2018,1512031508289527,398



Protocol metadata for largest apparent lines:


,PrimaryKey,Year,LineKey,n_pin_drops,PlotID,ProjectKey,DBKey,source,FormType,Direction,Measure,LineLengthAmount,SpacingIntervalAmount,SpacingType,n_distinct_pointnbr,min_pointnbr,max_pointnbr,n_raw_layer_records,n_distinct_layers,layers_present
0,15120315082630972018-06-26,2018,1512031508283479,400,CPER,NWERN_CPER,REPORT26Jun18CPERDIMA5.5aasof2020-06-26,AIM,NA,60,1,100,25,cm,400,1.0,400.0,1161,5,"Lower1, Lower2, Lower3, SoilSurface, TopCanopy"
1,17062316455172542019-12-10,2019,1706231645524497,400,El Reno,NWERN_ElReno,REPORT10Dec19ElRenoDIMA5.5aasof2020-06-26,AIM,NA,120,1,100,25,cm,400,1.0,400.0,925,4,"Lower1, Lower2, SoilSurface, TopCanopy"
2,17062316455172542019-12-10,2019,1706231645526272,400,El Reno,NWERN_ElReno,REPORT10Dec19ElRenoDIMA5.5aasof2020-06-26,AIM,NA,60,1,100,25,cm,400,1.0,400.0,996,5,"Lower1, Lower2, Lower4, SoilSurface, TopCanopy"
3,17062316455172542019-12-10,2019,1706231645528128,400,El Reno,NWERN_ElReno,REPORT10Dec19ElRenoDIMA5.5aasof2020-06-26,AIM,NA,0,1,100,25,cm,400,1.0,400.0,884,4,"Lower1, Lower2, SoilSurface, TopCanopy"
4,15120315082630972019-05-15,2019,1512031508289527,399,CPER,NWERN_CPER,REPORT15May19CPERDIMA5.5aasof2020-06-26,AIM,NA,0,1,100,25,cm,399,1.0,400.0,1138,6,"Lower1, Lower2, Lower3, Lower4, SoilSurface, T..."
5,15120315082630972018-06-26,2018,1512031508281623,398,CPER,NWERN_CPER,REPORT26Jun18CPERDIMA5.5aasof2020-06-26,AIM,NA,0,1,100,25,cm,398,1.0,400.0,1177,5,"Lower1, Lower2, Lower3, SoilSurface, TopCanopy"
6,15120315082630972018-06-26,2018,1512031508281623,398,CPER,NWERN_CPER,REPORT26Jun18CPERDIMA5.5aasof2020-06-26,AIM,NA,120,1,100,25,cm,398,1.0,400.0,1177,5,"Lower1, Lower2, Lower3, SoilSurface, TopCanopy"
7,15120315082630972018-06-26,2018,1512031508289527,398,CPER,NWERN_CPER,REPORT26Jun18CPERDIMA5.5aasof2020-06-26,AIM,NA,0,1,100,25,cm,398,1.0,400.0,1123,5,"Lower1, Lower2, Lower3, SoilSurface, TopCanopy"
8,15120315082630972019-05-15,2019,1512031508281623,398,CPER,NWERN_CPER,REPORT15May19CPERDIMA5.5aasof2020-06-26,AIM,NA,0,1,100,25,cm,398,1.0,400.0,1163,4,"Lower1, Lower2, SoilSurface, TopCanopy"
9,15120315082630972021-08-25,2021,1512031508281623,398,CPER,NWERN_CPER,NWERN_CPER_2025-10-01,DIMA,NA,120,1,100,25,cm,398,1.0,400.0,1197,4,"Lower1, Lower2, SoilSurface, TopCanopy"



PointNbr structure for 10 largest lines:


,PrimaryKey,Year,LineKey,n_distinct_points,min_pointnbr,max_pointnbr,n_numeric_pointnbr,point_numbers
0,15120315082630972018-06-26,2018,1512031508283479,400,1.0,400.0,400,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
1,17062316455172542019-12-10,2019,1706231645528128,400,1.0,400.0,400,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
2,17062316455172542019-12-10,2019,1706231645526272,400,1.0,400.0,400,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
3,17062316455172542019-12-10,2019,1706231645524497,400,1.0,400.0,400,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
4,15120315082630972019-05-15,2019,1512031508289527,399,1.0,400.0,399,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
5,15120315082630972019-05-15,2019,1512031508281623,398,1.0,400.0,398,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
6,19020816041349872022-02-15,2022,190208160414117,398,1.0,400.0,398,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
7,19020816041349872021-03-29,2021,190208160414117,398,1.0,400.0,398,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
8,15120315082630972018-06-26,2018,1512031508289527,398,1.0,400.0,398,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
9,19020816041349872020-10-19,2020,190208160414117,398,1.0,400.0,398,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Lines >200 apparent pin drops, summarized by protocol:


,source,DBKey,FormType,LineLengthAmount,SpacingIntervalAmount,SpacingType,n_lines,min_pin_drops,median_pin_drops,max_pin_drops
0,AIM,REPORT10Dec19ElRenoDIMA5.5aasof2020-06-26,NA,100,25,cm,3,400,400.0,400
1,AIM,REPORT26Jun18CPERDIMA5.5aasof2020-06-26,NA,100,25,cm,3,398,398.0,400
2,AIM,REPORT15May19CPERDIMA5.5aasof2020-06-26,NA,100,25,cm,3,397,398.0,399
3,DIMA,NWERN_CPER_2025-10-01,NA,100,25,cm,27,395,397.0,398
4,DIMA,NWERN_Akron_2025-10-01,NA,100,25,cm,15,396,396.0,398
...,...,...,...,...,...,...,...,...,...,...
101,AIM,REPORT17Sept20HAFBDIMA5.4asof2019-04-19,NA,100,25,cm,3,395,395.0,395
102,AIM,REPORT6Jul18PullmanDIMA5.5aasof2020-06-26,NA,100,25,cm,3,395,395.0,395
103,AIM,REPORT4Sept19PullmanDIMA5.5aasof2020-06-26,NA,100,25,cm,3,395,395.0,395
104,AIM,REPORT13May19PullmanDIMA5.5aasof2020-06-26,NA,100,25,cm,3,378,380.0,382


In [25]:
# ============================================================================
# 8. COMPARE OBSERVED PIN DROPS TO METADATA-EXPECTED PIN DROPS
#
# IMPORTANT:
#
#   observed_pins =
#       actual unique PrimaryKey × Year × LineKey × PointNbr records
#
#   expected_pins =
#       survey-design expectation derived from:
#
#           LineLengthAmount / SpacingIntervalAmount
#
#       after converting spacing to meters.
#
#   We are NOT replacing the observed denominator yet.
#   This cell is diagnostic only.
#
# Metadata are pulled from lpi_raw because lpi_valid intentionally carries
# only the composition-processing fields.
# ============================================================================


# ============================================================================
# A. DEFINE LINES THAT SURVIVED ALL VISIT-LEVEL FILTERS
# ============================================================================

con.execute("""
CREATE OR REPLACE TEMP TABLE valid_lines AS

SELECT DISTINCT
    PrimaryKey,
    Year,
    LineKey

FROM pins

WHERE PrimaryKey IS NOT NULL
  AND Year IS NOT NULL
  AND LineKey IS NOT NULL
;
""")


# ============================================================================
# B. OBSERVED UNIQUE PIN DROPS PER LINE
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE line_observed_pins AS

SELECT
    PrimaryKey,
    Year,
    LineKey,

    COUNT(*) AS observed_pins,

    MIN(
        TRY_CAST(PointNbr AS DOUBLE)
    ) AS min_pointnbr,

    MAX(
        TRY_CAST(PointNbr AS DOUBLE)
    ) AS max_pointnbr,

    COUNT(
        DISTINCT TRY_CAST(PointNbr AS DOUBLE)
    ) AS n_numeric_pointnbr

FROM pins

GROUP BY
    PrimaryKey,
    Year,
    LineKey
;
""")


# ============================================================================
# C. RETRIEVE RAW SURVEY-DESIGN METADATA FOR VALID LINES
#
# We retain consistency counts because one apparent line should not have
# multiple conflicting values for length or spacing.
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE line_metadata AS

SELECT
    r.PrimaryKey,
    TRY_CAST(r.SampleYear AS INTEGER) AS Year,
    r.LineKey,

    MAX(
        TRY_CAST(r.LineLengthAmount AS DOUBLE)
    ) AS line_length,

    MAX(
        TRY_CAST(r.SpacingIntervalAmount AS DOUBLE)
    ) AS spacing_interval,

    MAX(
        LOWER(TRIM(r.SpacingType))
    ) AS spacing_unit,

    MAX(
        TRY_CAST(r.Measure AS DOUBLE)
    ) AS measure,

    MAX(
        TRIM(r.Direction)
    ) AS direction,

    MAX(
        COALESCE(
            NULLIF(TRIM(r."source.x"), ''),
            NULLIF(TRIM(r."source.y"), '')
        )
    ) AS source,

    MAX(
        COALESCE(
            NULLIF(TRIM(r."DBKey.x"), ''),
            NULLIF(TRIM(r."DBKey.y"), '')
        )
    ) AS DBKey,

    MAX(
        COALESCE(
            NULLIF(TRIM(r."ProjectKey.x"), ''),
            NULLIF(TRIM(r."ProjectKey.y"), '')
        )
    ) AS ProjectKey,

    COUNT(
        DISTINCT r.LineLengthAmount
    ) AS n_line_length_values,

    COUNT(
        DISTINCT r.SpacingIntervalAmount
    ) AS n_spacing_values,

    COUNT(
        DISTINCT r.SpacingType
    ) AS n_spacing_units

FROM lpi_raw r

INNER JOIN valid_lines v
    ON r.PrimaryKey = v.PrimaryKey
   AND TRY_CAST(r.SampleYear AS INTEGER) = v.Year
   AND r.LineKey = v.LineKey

GROUP BY
    r.PrimaryKey,
    TRY_CAST(r.SampleYear AS INTEGER),
    r.LineKey
;
""")


# ============================================================================
# D. CALCULATE EXPECTED NUMBER OF PIN POSITIONS
#
# Based on the examples examined so far:
#
#   LineLengthAmount = 100
#   SpacingIntervalAmount = 25
#   SpacingType = cm
#
# gives:
#
#   100 m / 0.25 m = 400 expected positions
#
# We therefore treat LineLengthAmount as meters here and explicitly convert
# the spacing interval to meters.
#
# Unknown spacing units are NOT guessed.
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE line_denominator_qa AS

WITH converted AS (

    SELECT
        m.*,

        CASE
            WHEN spacing_interval IS NULL
            THEN NULL

            WHEN spacing_unit IN (
                'm',
                'meter',
                'meters'
            )
            THEN spacing_interval

            WHEN spacing_unit IN (
                'cm',
                'centimeter',
                'centimeters'
            )
            THEN spacing_interval / 100.0

            WHEN spacing_unit IN (
                'mm',
                'millimeter',
                'millimeters'
            )
            THEN spacing_interval / 1000.0

            WHEN spacing_unit IN (
                'ft',
                'foot',
                'feet'
            )
            THEN spacing_interval * 0.3048

            ELSE NULL
        END AS spacing_m

    FROM line_metadata m
),

expected AS (

    SELECT
        c.*,

        CASE
            WHEN line_length IS NOT NULL
             AND spacing_m IS NOT NULL
             AND spacing_m > 0

            THEN line_length / spacing_m

            ELSE NULL
        END AS expected_pins_raw

    FROM converted c
)

SELECT
    o.PrimaryKey,
    o.Year,
    o.LineKey,

    o.observed_pins,

    o.min_pointnbr,
    o.max_pointnbr,
    o.n_numeric_pointnbr,

    e.line_length,
    e.spacing_interval,
    e.spacing_unit,
    e.spacing_m,

    e.measure,
    e.direction,

    e.source,
    e.DBKey,
    e.ProjectKey,

    e.expected_pins_raw,

    ROUND(
        e.expected_pins_raw
    ) AS expected_pins,

    o.observed_pins
        - ROUND(e.expected_pins_raw)
        AS pin_difference,

    CASE
        WHEN e.expected_pins_raw > 0

        THEN
            100.0
            * o.observed_pins
            / e.expected_pins_raw

        ELSE NULL
    END AS pct_expected_observed,

    CASE
        WHEN e.expected_pins_raw > 0

        THEN
            100.0
            * (
                e.expected_pins_raw
                - o.observed_pins
              )
            / e.expected_pins_raw

        ELSE NULL
    END AS pct_expected_missing,

    e.n_line_length_values,
    e.n_spacing_values,
    e.n_spacing_units,

    CASE
        WHEN e.n_line_length_values > 1
          OR e.n_spacing_values > 1
          OR e.n_spacing_units > 1
        THEN TRUE
        ELSE FALSE
    END AS metadata_inconsistent,

    CASE
        WHEN e.expected_pins_raw IS NULL
        THEN 'NoExpectedCount'

        WHEN ABS(
            o.observed_pins
            - ROUND(e.expected_pins_raw)
        ) = 0
        THEN 'Exact'

        WHEN ABS(
            o.observed_pins
            - ROUND(e.expected_pins_raw)
        ) <= 2
        THEN 'Within2Pins'

        WHEN
            100.0
            * ABS(
                o.observed_pins
                - e.expected_pins_raw
              )
            / e.expected_pins_raw
            <= 5.0
        THEN 'Within5Percent'

        WHEN
            100.0
            * o.observed_pins
            / e.expected_pins_raw
            >= 90.0
        THEN '90to95PercentComplete'

        ELSE 'Review'
    END AS denominator_qa_class

FROM line_observed_pins o

LEFT JOIN expected e
    ON o.PrimaryKey = e.PrimaryKey
   AND o.Year = e.Year
   AND o.LineKey = e.LineKey
;
""")


# ============================================================================
# E. OVERALL QA SUMMARY
# ============================================================================

summary = con.execute("""
SELECT
    denominator_qa_class,

    COUNT(*) AS n_lines,

    MIN(pct_expected_observed)
        AS min_pct_expected_observed,

    MEDIAN(pct_expected_observed)
        AS median_pct_expected_observed,

    MAX(pct_expected_observed)
        AS max_pct_expected_observed

FROM line_denominator_qa

GROUP BY denominator_qa_class

ORDER BY
    CASE denominator_qa_class
        WHEN 'Exact' THEN 1
        WHEN 'Within2Pins' THEN 2
        WHEN 'Within5Percent' THEN 3
        WHEN '90to95PercentComplete' THEN 4
        WHEN 'Review' THEN 5
        WHEN 'NoExpectedCount' THEN 6
        ELSE 99
    END
;
""").df()

print("Observed vs metadata-expected pin counts:")
display(summary)


# ============================================================================
# F. INSPECT THE LARGEST LINES
# ============================================================================

largest_lines = con.execute("""
SELECT
    PrimaryKey,
    Year,
    LineKey,

    source,
    ProjectKey,

    observed_pins,
    expected_pins,

    pin_difference,
    pct_expected_observed,
    pct_expected_missing,

    min_pointnbr,
    max_pointnbr,

    line_length,
    spacing_interval,
    spacing_unit,

    denominator_qa_class

FROM line_denominator_qa

ORDER BY observed_pins DESC

LIMIT 100
;
""").df()

print("\nLargest apparent lines:")
display(largest_lines)


# ============================================================================
# G. SPECIFICALLY INSPECT INCOMPLETE 400-POSITION DESIGNS
#
# This will surface cases such as 355, 361, 362, 378, etc.
# ============================================================================

incomplete_400_lines = con.execute("""
SELECT
    PrimaryKey,
    Year,
    LineKey,

    source,
    DBKey,
    ProjectKey,

    observed_pins,
    expected_pins,

    pin_difference,
    pct_expected_observed,
    pct_expected_missing,

    min_pointnbr,
    max_pointnbr,
    n_numeric_pointnbr,

    line_length,
    spacing_interval,
    spacing_unit,

    denominator_qa_class

FROM line_denominator_qa

WHERE expected_pins BETWEEN 399 AND 401
  AND observed_pins < 395

ORDER BY observed_pins ASC
;
""").df()

print("\nIncomplete nominal 400-position lines:")
display(incomplete_400_lines)


# ============================================================================
# H. METADATA CONSISTENCY
# ============================================================================

metadata_qa = con.execute("""
SELECT
    COUNT(*) AS n_lines,

    SUM(
        CASE
            WHEN metadata_inconsistent
            THEN 1
            ELSE 0
        END
    ) AS n_metadata_inconsistent,

    SUM(
        CASE
            WHEN expected_pins IS NULL
            THEN 1
            ELSE 0
        END
    ) AS n_no_expected_count

FROM line_denominator_qa
;
""").df()

print("\nMetadata QA:")
display(metadata_qa)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Observed vs metadata-expected pin counts:


,denominator_qa_class,n_lines,min_pct_expected_observed,median_pct_expected_observed,max_pct_expected_observed
0,Exact,112390,99.568,100.000,100.609756
1,Within2Pins,34894,95.504,101.600,104.166667
2,Within5Percent,385,95.000,99.000,99.250000
3,90to95PercentComplete,294,90.000,93.472,200.000000
4,Review,2359,2.000,71.120,89.408000
5,NoExpectedCount,27,NaN,NaN,NaN


...

,PrimaryKey,Year,LineKey,source,ProjectKey,observed_pins,expected_pins,pin_difference,pct_expected_observed,pct_expected_missing,min_pointnbr,max_pointnbr,line_length,spacing_interval,spacing_unit,denominator_qa_class
0,17062316455172542019-12-10,2019,1706231645524497,AIM,NWERN_ElReno,400,400.0,0.0,100.00,0.00,1.0,400.0,100.0,25.0,cm,Exact
1,17062316455172542019-12-10,2019,1706231645526272,AIM,NWERN_ElReno,400,400.0,0.0,100.00,0.00,1.0,400.0,100.0,25.0,cm,Exact
2,17062316455172542019-12-10,2019,1706231645528128,AIM,NWERN_ElReno,400,400.0,0.0,100.00,0.00,1.0,400.0,100.0,25.0,cm,Exact
3,15120315082630972018-06-26,2018,1512031508283479,AIM,NWERN_CPER,400,400.0,0.0,100.00,0.00,1.0,400.0,100.0,25.0,cm,Exact
4,15120315082630972019-05-15,2019,1512031508289527,AIM,NWERN_CPER,399,400.0,-1.0,99.75,0.25,1.0,400.0,100.0,25.0,cm,Within2Pins
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,15052012001494532023-06-21,2023,1505201200212620,DIMA,NWERN_HAFB,396,400.0,-4.0,99.00,1.00,1.0,400.0,100.0,25.0,cm,Within5Percent
96,20082716121036612021-06-17,2021,2008271612115997,AIM,NWERN_Lordsburg,396,400.0,-4.0,99.00,1.00,1.0,400.0,100.0,25.0,cm,Within5Percent
97,19020816041349872021-12-08,2021,1902081604141973,DIMA,NWERN_Akron,396,400.0,-4.0,99.00,1.00,1.0,400.0,100.0,25.0,cm,Within5Percent
98,15050113465465692020-09-15,2020,1505011346563464,AIM,NWERN_JER,396,400.0,-4.0,99.00,1.00,1.0,400.0,100.0,25.0,cm,Within5Percent



Incomplete nominal 400-position lines:


,PrimaryKey,Year,LineKey,source,DBKey,ProjectKey,observed_pins,expected_pins,pin_difference,pct_expected_observed,pct_expected_missing,min_pointnbr,max_pointnbr,n_numeric_pointnbr,line_length,spacing_interval,spacing_unit,denominator_qa_class
0,19081514072945942021-07-19,2021,1908151407316042,AIM,REPORT19Jul21MortonDIMA5.5aasof2020-06-26,NWERN_Morton,98,400.0,-302.0,24.50,75.50,4.0,400.0,98,100.0,25.00,cm,Review
1,19081514072945942019-10-31,2019,1908151407316042,AIM,REPORT31Oct19MortonDIMA5.5aasof2020-06-26,NWERN_Morton,99,400.0,-301.0,24.75,75.25,4.0,400.0,99,100.0,25.00,cm,Review
2,19081514072945942019-10-31,2019,1908151407312091,AIM,REPORT31Oct19MortonDIMA5.5aasof2020-06-26,NWERN_Morton,99,400.0,-301.0,24.75,75.25,4.0,400.0,99,100.0,25.00,cm,Review
3,19081514072945942019-10-31,2019,1908151407314186,AIM,REPORT31Oct19MortonDIMA5.5aasof2020-06-26,NWERN_Morton,99,400.0,-301.0,24.75,75.25,4.0,400.0,99,100.0,25.00,cm,Review
4,19081514072945942021-07-19,2021,1908151407312091,AIM,REPORT19Jul21MortonDIMA5.5aasof2020-06-26,NWERN_Morton,99,400.0,-301.0,24.75,75.25,4.0,400.0,99,100.0,25.00,cm,Review
5,15080712440114922022-09-28,2022,1508071244076665,DIMA,NWERN_Mandan_2025-10-01,NWERN_Mandan,99,400.0,-301.0,24.75,75.25,1.0,100.0,99,100.0,25.00,cm,Review
6,19081514072945942021-07-19,2021,1908151407314186,AIM,REPORT19Jul21MortonDIMA5.5aasof2020-06-26,NWERN_Morton,99,400.0,-301.0,24.75,75.25,4.0,400.0,99,100.0,25.00,cm,Review
7,17030610202051092020-10-05,2020,1703061020229787,DIMA,REPORT5Oct20PullmanDIMA5.5aasof2020-06-26,NWERN_Pullman,355,400.0,-45.0,88.75,11.25,1.0,400.0,355,100.0,25.00,cm,Review
8,17030610202051092020-10-05,2020,1703061020221564,DIMA,REPORT5Oct20PullmanDIMA5.5aasof2020-06-26,NWERN_Pullman,361,400.0,-39.0,90.25,9.75,1.0,400.0,361,100.0,25.00,cm,90to95PercentComplete
9,17030610202051092020-10-05,2020,1703061020223420,DIMA,REPORT5Oct20PullmanDIMA5.5aasof2020-06-26,NWERN_Pullman,362,400.0,-38.0,90.50,9.50,1.0,400.0,362,100.0,25.00,cm,90to95PercentComplete



Metadata QA:


,n_lines,n_metadata_inconsistent,n_no_expected_count
0,150349,0.0,27.0


In [26]:
line_count_per_visit = con.execute("""
SELECT
    PrimaryKey,
    Year,
    COUNT(DISTINCT LineKey) AS n_lines

FROM pins

GROUP BY
    PrimaryKey,
    Year
;
""").df()

display(
    con.execute("""
        SELECT
            COUNT(*) AS n_visits,
            MIN(n_lines) AS min_lines,
            MEDIAN(n_lines) AS median_lines,
            MAX(n_lines) AS max_lines,
            AVG(n_lines) AS mean_lines
        FROM (
            SELECT
                PrimaryKey,
                Year,
                COUNT(DISTINCT LineKey) AS n_lines
            FROM pins
            GROUP BY PrimaryKey, Year
        )
    """).df()
)

display(
    con.execute("""
        SELECT
            n_lines,
            COUNT(*) AS n_visits
        FROM (
            SELECT
                PrimaryKey,
                Year,
                COUNT(DISTINCT LineKey) AS n_lines
            FROM pins
            GROUP BY PrimaryKey, Year
        )
        GROUP BY n_lines
        ORDER BY n_lines
    """).df()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_visits,min_lines,median_lines,max_lines,mean_lines
0,50626,1,3.0,20,2.969798


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_lines,n_visits
0,1,592
1,2,18793
2,3,30154
3,4,1
4,5,1
5,20,1085


In [27]:
# ============================================================================
# DIAGNOSE VISITS WITH >3 LINES
#
# Goal:
#   Identify the source, project, database, protocol metadata, and spatial
#   distribution of visits containing 20 lines.
#
# READ ONLY:
#   No visits or lines are removed here.
# ============================================================================


# ============================================================================
# A. IDENTIFY VISITS WITH MORE THAN 3 LINES
# ============================================================================

con.execute("""
CREATE OR REPLACE TEMP TABLE visits_gt3_lines AS

SELECT
    PrimaryKey,
    Year,
    COUNT(DISTINCT LineKey) AS n_lines

FROM pins

GROUP BY
    PrimaryKey,
    Year

HAVING COUNT(DISTINCT LineKey) > 3
;
""")


print("Visits with >3 lines:")
display(
    con.execute("""
        SELECT
            n_lines,
            COUNT(*) AS n_visits

        FROM visits_gt3_lines

        GROUP BY n_lines
        ORDER BY n_lines
    """).df()
)


# ============================================================================
# B. GET VISIT-LEVEL SOURCE / PROTOCOL METADATA
# ============================================================================

gt3_visit_metadata = con.execute("""
SELECT
    v.PrimaryKey,
    v.Year,
    v.n_lines,

    MAX(r.PlotID) AS PlotID,

    MAX(
        COALESCE(
            NULLIF(TRIM(r."ProjectKey.x"), ''),
            NULLIF(TRIM(r."ProjectKey.y"), '')
        )
    ) AS ProjectKey,

    MAX(
        COALESCE(
            NULLIF(TRIM(r."DBKey.x"), ''),
            NULLIF(TRIM(r."DBKey.y"), '')
        )
    ) AS DBKey,

    MAX(
        COALESCE(
            NULLIF(TRIM(r."source.x"), ''),
            NULLIF(TRIM(r."source.y"), '')
        )
    ) AS source,

    MAX(r.FormType) AS FormType,

    MAX(
        TRY_CAST(r.Latitude_NAD83 AS DOUBLE)
    ) AS Latitude_NAD83,

    MAX(
        TRY_CAST(r.Longitude_NAD83 AS DOUBLE)
    ) AS Longitude_NAD83,

    MAX(r.EcologicalSiteID) AS EcologicalSiteID,

    COUNT(DISTINCT r.LineKey)
        AS raw_n_lines,

    COUNT(DISTINCT r.LineLengthAmount)
        AS n_line_length_values,

    MIN(
        TRY_CAST(r.LineLengthAmount AS DOUBLE)
    ) AS min_line_length,

    MAX(
        TRY_CAST(r.LineLengthAmount AS DOUBLE)
    ) AS max_line_length,

    COUNT(DISTINCT r.SpacingIntervalAmount)
        AS n_spacing_values,

    MIN(
        TRY_CAST(r.SpacingIntervalAmount AS DOUBLE)
    ) AS min_spacing,

    MAX(
        TRY_CAST(r.SpacingIntervalAmount AS DOUBLE)
    ) AS max_spacing,

    STRING_AGG(
        DISTINCT r.SpacingType,
        ', '
        ORDER BY r.SpacingType
    ) AS spacing_units

FROM visits_gt3_lines v

INNER JOIN lpi_raw r
    ON r.PrimaryKey = v.PrimaryKey
   AND TRY_CAST(r.SampleYear AS INTEGER) = v.Year

GROUP BY
    v.PrimaryKey,
    v.Year,
    v.n_lines

ORDER BY
    v.n_lines DESC,
    v.Year,
    v.PrimaryKey
;
""").df()


print("\nVisit-level metadata for >3-line visits:")
display(gt3_visit_metadata)


# ============================================================================
# C. SUMMARIZE BY SOURCE / PROJECT / DATABASE
# ============================================================================

gt3_source_summary = con.execute("""
WITH visit_metadata AS (

    SELECT
        v.PrimaryKey,
        v.Year,
        v.n_lines,

        MAX(
            COALESCE(
                NULLIF(TRIM(r."ProjectKey.x"), ''),
                NULLIF(TRIM(r."ProjectKey.y"), '')
            )
        ) AS ProjectKey,

        MAX(
            COALESCE(
                NULLIF(TRIM(r."DBKey.x"), ''),
                NULLIF(TRIM(r."DBKey.y"), '')
            )
        ) AS DBKey,

        MAX(
            COALESCE(
                NULLIF(TRIM(r."source.x"), ''),
                NULLIF(TRIM(r."source.y"), '')
            )
        ) AS source,

        MAX(r.FormType) AS FormType

    FROM visits_gt3_lines v

    INNER JOIN lpi_raw r
        ON r.PrimaryKey = v.PrimaryKey
       AND TRY_CAST(r.SampleYear AS INTEGER) = v.Year

    GROUP BY
        v.PrimaryKey,
        v.Year,
        v.n_lines
)

SELECT
    source,
    ProjectKey,
    DBKey,
    FormType,
    n_lines,

    COUNT(*) AS n_visits,

    MIN(Year) AS first_year,
    MAX(Year) AS last_year

FROM visit_metadata

GROUP BY
    source,
    ProjectKey,
    DBKey,
    FormType,
    n_lines

ORDER BY
    n_visits DESC
;
""").df()


print("\n>3-line visits summarized by source:")
display(gt3_source_summary)


# ============================================================================
# D. LINE-LEVEL STRUCTURE OF THE 20-LINE VISITS
# ============================================================================

gt3_line_structure = con.execute("""
SELECT
    p.PrimaryKey,
    p.Year,
    p.LineKey,

    COUNT(*) AS n_pin_drops,

    MIN(
        TRY_CAST(p.PointNbr AS DOUBLE)
    ) AS min_pointnbr,

    MAX(
        TRY_CAST(p.PointNbr AS DOUBLE)
    ) AS max_pointnbr

FROM pins p

INNER JOIN visits_gt3_lines v
    ON p.PrimaryKey = v.PrimaryKey
   AND p.Year = v.Year

GROUP BY
    p.PrimaryKey,
    p.Year,
    p.LineKey

ORDER BY
    p.PrimaryKey,
    p.Year,
    p.LineKey
;
""").df()


print("\nLine structure within >3-line visits:")
display(gt3_line_structure)


# ============================================================================
# E. PIN COUNTS PER >3-LINE VISIT
# ============================================================================

gt3_denominator_summary = con.execute("""
SELECT
    v.PrimaryKey,
    v.Year,
    v.n_lines,

    COUNT(p.PointNbr) AS n_pin_drops

FROM visits_gt3_lines v

INNER JOIN pins p
    ON v.PrimaryKey = p.PrimaryKey
   AND v.Year = p.Year

GROUP BY
    v.PrimaryKey,
    v.Year,
    v.n_lines

ORDER BY
    n_pin_drops DESC
;
""").df()


print("\nTotal pin denominator for >3-line visits:")
display(gt3_denominator_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Visits with >3 lines:


,n_lines,n_visits
0,4,1
1,5,1
2,20,1085



Visit-level metadata for >3-line visits:


,PrimaryKey,Year,n_lines,PlotID,ProjectKey,DBKey,source,FormType,Latitude_NAD83,Longitude_NAD83,EcologicalSiteID,raw_n_lines,n_line_length_values,min_line_length,max_line_length,n_spacing_values,min_spacing,max_spacing,spacing_units
0,000003_2018-07-09_0.336_36.9_l,2018,20,000003,LandPKS,LandPKS_2025-05-09,LandPKS,LPI,0.33600,36.90000,NA,20,1,1.0,1.0,1,20.0,20.0,cm
1,003klfv_2018-04-30_-3.03292_39.89712_s,2018,20,003klfv,LandPKS,LandPKS_2025-05-09,LandPKS,LPI,-3.03292,39.89712,NA,20,1,1.0,1.0,1,20.0,20.0,cm
2,003klfv_2018-05-29_-3.03292_39.89712_s,2018,20,003klfv,LandPKS,LandPKS_2025-05-09,LandPKS,LPI,-3.03292,39.89712,NA,20,1,1.0,1.0,1,20.0,20.0,cm
3,006_TTC-Mtt_2018-05-24_-3.49884_38.40132_m,2018,20,006 TTC-Mtt,LandPKS,LandPKS_2025-05-09,LandPKS,LPI,-3.49884,38.40132,NA,20,1,1.0,1.0,1,20.0,20.0,cm
4,006_TTC-voi_2018-05-21_-3.38446_38.57406_m,2018,20,006 TTC-voi,LandPKS,LandPKS_2025-05-09,LandPKS,LPI,-3.38446,38.57406,NA,20,1,1.0,1.0,1,20.0,20.0,cm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082,walaweyn_9_2024-05-17_2.59088_44.8631_s,2024,20,walaweyn 9,LandPKS,LandPKS_2025-05-09,LandPKS,LPI,2.59088,44.86310,NA,20,1,1.0,1.0,1,20.0,20.0,cm
1083,wanlaweyn_2024-02-21_2.57777_44.85789_s,2024,20,wanlaweyn,LandPKS,LandPKS_2025-05-09,LandPKS,LPI,2.57777,44.85789,NA,20,1,1.0,1.0,1,20.0,20.0,cm
1084,wanlweyn_2024-02-19_2.57859_44.85788_s,2024,20,wanlweyn,LandPKS,LandPKS_2025-05-09,LandPKS,LPI,2.57859,44.85788,NA,20,1,1.0,1.0,1,20.0,20.0,cm
1085,18071612562728672018-09-01,2018,5,MOOSETRACK_EXCLOSURE,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,TERRADAT,NA,61.16225,-149.78677,UNKNOWN,5,2,14.0,25.0,1,50.0,50.0,cm


...

,source,ProjectKey,DBKey,FormType,n_lines,n_visits,first_year,last_year
0,LandPKS,LandPKS,LandPKS_2025-05-09,LPI,20,1085,2018,2024
1,TERRADAT,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,NA,5,1,2018,2018
2,TERRADAT,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,NA,4,1,2018,2018



Line structure within >3-line visits:


,PrimaryKey,Year,LineKey,n_pin_drops,min_pointnbr,max_pointnbr
0,000003_2018-07-09_0.336_36.9_l,2018,EAST_10m,5,1.0,5.0
1,000003_2018-07-09_0.336_36.9_l,2018,EAST_15m,5,1.0,5.0
2,000003_2018-07-09_0.336_36.9_l,2018,EAST_20m,5,1.0,5.0
3,000003_2018-07-09_0.336_36.9_l,2018,EAST_25m,5,1.0,5.0
4,000003_2018-07-09_0.336_36.9_l,2018,EAST_5m,5,1.0,5.0
...,...,...,...,...,...,...
21704,zoona_dhabab_gallacyo_2022-03-22_6.71931_47.42...,2022,WEST_10m,5,1.0,5.0
21705,zoona_dhabab_gallacyo_2022-03-22_6.71931_47.42...,2022,WEST_15m,5,1.0,5.0
21706,zoona_dhabab_gallacyo_2022-03-22_6.71931_47.42...,2022,WEST_20m,5,1.0,5.0
21707,zoona_dhabab_gallacyo_2022-03-22_6.71931_47.42...,2022,WEST_25m,5,1.0,5.0



Total pin denominator for >3-line visits:


,PrimaryKey,Year,n_lines,n_pin_drops
0,18071712175618342018-09-01,2018,4,160
1,18071612562728672018-09-01,2018,5,145
2,IsakomallaM3_2024-06-21_3.272808_37.020318_h,2024,20,100
3,BelgeshT2_2022-12-13_0.841764_38.716251_h,2022,20,100
4,Qotan_Area_1_2022-03-27_9.56164_50.49274_s,2022,20,100
...,...,...,...,...
1082,36B_Biomass_2019-10-16_-0.7563_35.1938_c,2019,20,100
1083,Lerata_b_nongoroshi_inside_Boma_2024-07-25_0.7...,2024,20,100
1084,RT-_8_2022-05-13_-10.79944_-76.40788_2,2022,20,100
1085,MARTISBRV_2018-03-16_1.3893_36.6348_a,2018,20,100


In [28]:
# ============================================================================
# 6C. EXCLUDE NONSTANDARD >3-LINE VISITS AND CREATE lpi_valid_final
#
# Target sampling unit:
#   PrimaryKey × Year visits with no more than 3 distinct LPI lines.
#
# Rationale:
#   - 1,085 twenty-line visits are LandPKS and represent a fundamentally
#     different sampling design.
#   - the remaining 4-line and 5-line visits are rare nonstandard AIM
#     exclosure designs.
#   - visits are excluded in full; lines are never subsampled.
#
# Input:
#   lpi_valid
#
# Output:
#   excluded_nonstandard_line_visits
#   lpi_valid_final
#
# Cell 7 should use lpi_valid_final.
# ============================================================================


# ============================================================================
# A. IDENTIFY VISITS WITH >3 DISTINCT LINES
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE excluded_nonstandard_line_visits AS

SELECT
    PrimaryKey,
    Year,
    COUNT(DISTINCT LineKey) AS n_lines

FROM lpi_valid

WHERE PrimaryKey IS NOT NULL
  AND Year IS NOT NULL
  AND LineKey IS NOT NULL

GROUP BY
    PrimaryKey,
    Year

HAVING COUNT(DISTINCT LineKey) > 3
;
""")


# ============================================================================
# B. QA: HOW MANY VISITS ARE BEING EXCLUDED?
# ============================================================================

line_exclusion_qa = con.execute("""
SELECT
    n_lines,
    COUNT(*) AS n_excluded_visits

FROM excluded_nonstandard_line_visits

GROUP BY n_lines

ORDER BY n_lines
;
""").df()

print("Nonstandard line-count exclusions:")
display(line_exclusion_qa)


# ============================================================================
# C. QA: SOURCE / PROJECT OF EXCLUDED VISITS
#
# This should make the LandPKS 20-line design obvious while retaining the
# two rare >3-line BLM AIM/TERRADAT visits as separate exclusions.
# ============================================================================

excluded_line_source_qa = con.execute("""
WITH visit_source AS (

    SELECT
        x.PrimaryKey,
        x.Year,
        x.n_lines,

        MAX(l.ProjectKey) AS ProjectKey,
        MAX(l.DBKey) AS DBKey

    FROM excluded_nonstandard_line_visits x

    INNER JOIN lpi_valid l
        ON x.PrimaryKey = l.PrimaryKey
       AND x.Year = l.Year

    GROUP BY
        x.PrimaryKey,
        x.Year,
        x.n_lines
)

SELECT
    n_lines,
    ProjectKey,
    DBKey,
    COUNT(*) AS n_excluded_visits

FROM visit_source

GROUP BY
    n_lines,
    ProjectKey,
    DBKey

ORDER BY
    n_lines DESC,
    n_excluded_visits DESC
;
""").df()

print("\nExcluded visits by line count / project / database:")
display(excluded_line_source_qa)


# ============================================================================
# D. CREATE FINAL RETAINED TALL LPI TABLE
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE lpi_valid_final AS

SELECT l.*

FROM lpi_valid l

LEFT JOIN excluded_nonstandard_line_visits x
    ON l.PrimaryKey = x.PrimaryKey
   AND l.Year = x.Year

WHERE x.PrimaryKey IS NULL
;
""")


# ============================================================================
# E. FINAL VISIT ACCOUNTING
# ============================================================================

final_line_filter_qa = con.execute("""
SELECT

    (
        SELECT COUNT(DISTINCT (PrimaryKey, Year))
        FROM lpi_valid
        WHERE PrimaryKey IS NOT NULL
          AND Year IS NOT NULL
    ) AS n_visits_before_line_filter,

    (
        SELECT COUNT(*)
        FROM excluded_nonstandard_line_visits
    ) AS n_excluded_nonstandard_line_visits,

    (
        SELECT COUNT(DISTINCT (PrimaryKey, Year))
        FROM lpi_valid_final
        WHERE PrimaryKey IS NOT NULL
          AND Year IS NOT NULL
    ) AS n_visits_after_line_filter,

    (
        SELECT COUNT(*)
        FROM lpi_valid
    ) AS n_rows_before_line_filter,

    (
        SELECT COUNT(*)
        FROM lpi_valid_final
    ) AS n_rows_after_line_filter

;
""").df()

print("\nFinal line-count filter accounting:")
display(final_line_filter_qa)


# ============================================================================
# F. YEARLY RETAINED VISITS
# ============================================================================

final_year_qa = con.execute("""
SELECT
    Year,
    COUNT(DISTINCT PrimaryKey) AS n_retained_plot_visits

FROM lpi_valid_final

WHERE PrimaryKey IS NOT NULL
  AND Year IS NOT NULL

GROUP BY Year

ORDER BY Year
;
""").df()

print("\nFinal retained visits by year:")
display(final_year_qa)


# ============================================================================
# G. HARD GUARD: NO >3-LINE VISITS MAY REMAIN
# ============================================================================

n_bad_visits = con.execute("""
SELECT COUNT(*)

FROM (

    SELECT
        PrimaryKey,
        Year,
        COUNT(DISTINCT LineKey) AS n_lines

    FROM lpi_valid_final

    WHERE PrimaryKey IS NOT NULL
      AND Year IS NOT NULL
      AND LineKey IS NOT NULL

    GROUP BY
        PrimaryKey,
        Year

    HAVING COUNT(DISTINCT LineKey) > 3

) x
;
""").fetchone()[0]

assert n_bad_visits == 0, (
    f"{n_bad_visits:,} visits with >3 lines remain in lpi_valid_final."
)


print()
print("PASS: lpi_valid_final created.")
print(
    "All retained visits now contain no more than 3 distinct LPI lines."
)

Nonstandard line-count exclusions:


,n_lines,n_excluded_visits
0,4,1
1,5,1
2,20,1085



Excluded visits by line count / project / database:


,n_lines,ProjectKey,DBKey,n_excluded_visits
0,20,LandPKS,LandPKS_2025-05-09,1085
1,5,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,1
2,4,BLM_AIM,BLM_Natl_AIM_TerrADat_Public,1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...


Final line-count filter accounting:


,n_visits_before_line_filter,n_excluded_nonstandard_line_visits,n_visits_after_line_filter,n_rows_before_line_filter,n_rows_after_line_filter
0,50626,1087,49539,15875184,15660061



Final retained visits by year:


,Year,n_retained_plot_visits
0,2018,6944
1,2019,8761
2,2020,7206
3,2021,8454
4,2022,8882
5,2023,8083
6,2024,1201
7,2025,8



PASS: lpi_valid_final created.
All retained visits now contain no more than 3 distinct LPI lines.


In [29]:
# ============================================================================
# 7. DEFINE UNIQUE PIN-DROP DENOMINATOR
#
# A pin drop is the fundamental sampling trial:
#
#     PrimaryKey × Year × LineKey × PointNbr
#
# This SAME denominator is used for:
#
#   1. top-canopy / first-hit species composition
#   2. top-canopy / first-hit functional-group composition
#   3. multilayer / any-hit species composition
#   4. multilayer / any-hit functional-group composition
#
#
# FIRST-HIT INTERPRETATION
# ------------------------
# Each pin contributes one first-hit outcome.
#
#
# MULTILAYER / ANY-HIT INTERPRETATION
# -----------------------------------
# Each pin is still ONE trial, but the trial may have MULTIPLE outcomes.
#
# Example:
#
#     TopCanopy = shrub
#     Lower1    = grass
#     Lower2    = forb
#
# That one pin contributes:
#
#     shrub = 1 successful pin
#     grass = 1 successful pin
#     forb  = 1 successful pin
#
# Therefore multilayer composition may sum to >100%.
#
# However, repeated occurrences of the SAME taxon or FG on one pin
# will later be deduplicated before counting:
#
#     shrub / shrub / grass
#
# contributes:
#
#     shrub = 1
#     grass = 1
#
# NOT shrub = 2.
#
#
# A line with 150 pin drops contributes 150 denominator units.
# The number of vertical hit records does not change that denominator.
# ============================================================================


# ============================================================================
# A. UNIQUE PIN TABLE
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE pins AS

SELECT DISTINCT
    PrimaryKey,
    Year,
    LineKey,
    PointNbr

FROM lpi_valid_final

WHERE PrimaryKey IS NOT NULL
  AND Year IS NOT NULL
  AND LineKey IS NOT NULL
  AND PointNbr IS NOT NULL
;
""")


# ============================================================================
# B. VISIT-LEVEL PIN DENOMINATOR
#
# n_points = number of unique pin-drop trials in the visit.
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE plot_totals AS

SELECT
    PrimaryKey,
    Year,
    COUNT(*) AS n_points

FROM pins

GROUP BY
    PrimaryKey,
    Year
;
""")


# ============================================================================
# C. OVERALL POINT-COUNT QA
# ============================================================================

point_qa = con.execute("""
SELECT
    COUNT(*) AS n_plot_years,
    MIN(n_points) AS min_points,
    MEDIAN(n_points) AS median_points,
    MAX(n_points) AS max_points,
    AVG(n_points) AS mean_points

FROM plot_totals
;
""").df()

print("Unique pin-drop denominator QA:")
display(point_qa)


# ============================================================================
# D. YEARLY DENOMINATOR QA
# ============================================================================

point_year_qa = con.execute("""
SELECT
    Year,

    COUNT(*) AS n_plot_visits,

    MIN(n_points) AS min_points,
    MEDIAN(n_points) AS median_points,
    MAX(n_points) AS max_points,
    AVG(n_points) AS mean_points

FROM plot_totals

GROUP BY Year
ORDER BY Year
;
""").df()

print("\nPin-drop denominator by year:")
display(point_year_qa)


# ============================================================================
# E. OPTIONAL LINE-LEVEL QA
#
# Verifies the actual number of pin drops represented on each line.
#
# A 150-pin line should return n_pin_drops = 150.
# ============================================================================

line_point_qa = con.execute("""
SELECT
    PrimaryKey,
    Year,
    LineKey,
    COUNT(*) AS n_pin_drops

FROM pins

GROUP BY
    PrimaryKey,
    Year,
    LineKey

ORDER BY
    n_pin_drops DESC,
    Year,
    PrimaryKey,
    LineKey
;
""").df()

print("\nLargest line-level pin counts:")
display(line_point_qa.head(25))


# ============================================================================
# F. HARD CONSISTENCY CHECK
#
# The sum of visit denominators must exactly equal the number of unique
# pin records in pins.
# ============================================================================

n_pins = con.execute("""
SELECT COUNT(*)
FROM pins
;
""").fetchone()[0]

sum_denominators = con.execute("""
SELECT SUM(n_points)
FROM plot_totals
;
""").fetchone()[0]

assert n_pins == sum_denominators, (
    f"Pin accounting mismatch: "
    f"{n_pins:,} unique pins vs "
    f"{sum_denominators:,} summed visit denominators."
)


# Every retained visit represented in pins should have exactly one
# plot_totals denominator row.

n_denominator_rows = con.execute("""
SELECT COUNT(*)
FROM plot_totals
;
""").fetchone()[0]

n_pin_visits = con.execute("""
SELECT COUNT(DISTINCT (PrimaryKey, Year))
FROM pins
;
""").fetchone()[0]

assert n_denominator_rows == n_pin_visits, (
    f"Visit denominator mismatch: "
    f"{n_denominator_rows:,} plot_totals rows vs "
    f"{n_pin_visits:,} visits represented in pins."
)


print()
print("PASS: common pin-drop denominator is established.")
print(
    "Both first-hit and multilayer composition will use "
    "plot_totals.n_points as the denominator."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique pin-drop denominator QA:


,n_plot_years,min_points,median_points,max_points,mean_points
0,49539,6,150.0,1200,135.857203



Pin-drop denominator by year:


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Largest line-level pin counts:


,PrimaryKey,Year,LineKey,n_pin_drops
0,15120315082630972018-06-26,2018,1512031508283479,400
1,17062316455172542019-12-10,2019,1706231645524497,400
2,17062316455172542019-12-10,2019,1706231645526272,400
3,17062316455172542019-12-10,2019,1706231645528128,400
4,15120315082630972019-05-15,2019,1512031508289527,399
5,15120315082630972018-06-26,2018,1512031508281623,398
6,15120315082630972018-06-26,2018,1512031508289527,398
7,19020816041349872018-12-21,2018,190208160414117,398
8,15120315082630972019-05-15,2019,1512031508281623,398
9,19020816041349872020-04-06,2020,190208160414117,398


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


PASS: common pin-drop denominator is established.
Both first-hit and multilayer composition will use plot_totals.n_points as the denominator.


# ============================================================================
# 8. PLOT-VISIT METADATA + UNKNOWN-CODE FLAG
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE plot_metadata AS
SELECT
    p.PrimaryKey,
    p.Year,
    MIN(l.DateVisited) AS DateVisited,
    MIN(l.DBKey) AS DBKey,
    MIN(l.ProjectKey) AS ProjectKey,
    MIN(l.source) AS source,
    MIN(l.Latitude_NAD83) AS Latitude_NAD83,
    MIN(l.Longitude_NAD83) AS Longitude_NAD83,
    p.n_points,

    u.n_all_lpi_hits,
    u.n_unknown_code_hits,
    u.unknown_code_pct_all_hits,
    u.unknown_code_gt5pct

FROM plot_totals p

LEFT JOIN lpi_valid l
    ON p.PrimaryKey = l.PrimaryKey
   AND p.Year = l.Year

LEFT JOIN visit_unknown_code_qa u
    ON p.PrimaryKey = u.PrimaryKey
   AND p.Year = u.Year

GROUP BY
    p.PrimaryKey,
    p.Year,
    p.n_points,
    u.n_all_lpi_hits,
    u.n_unknown_code_hits,
    u.unknown_code_pct_all_hits,
    u.unknown_code_gt5pct
;
""")

missing_unknown_qa = con.execute("""
SELECT COUNT(*)
FROM plot_metadata
WHERE unknown_code_pct_all_hits IS NULL
   OR unknown_code_gt5pct IS NULL
""").fetchone()[0]

assert missing_unknown_qa == 0, (
    "A retained plot visit is missing UnknownCode QA metadata."
)


In [ ]:
# ============================================================================
# 6A. QA — NO_CANOPY PINS MUST HAVE NO LOWER-LAYER CONTACTS
# ============================================================================

no_canopy_lower_violations = con.execute("""
WITH no_canopy_pins AS (

    SELECT DISTINCT
        PrimaryKey,
        Year,
        LineKey,
        PointNbr

    FROM lpi

    WHERE
        layer = 'TopCanopy'
        AND observed_code = '__NO_CANOPY__'
)

SELECT
    l.PrimaryKey,
    l.Year,
    l.LineKey,
    l.PointNbr,
    l.layer,
    l.observed_code

FROM lpi l

INNER JOIN no_canopy_pins n
    USING (
        PrimaryKey,
        Year,
        LineKey,
        PointNbr
    )

WHERE l.layer IN (
    'Lower1',
    'Lower2',
    'Lower3',
    'Lower4',
    'Lower5',
    'Lower6',
    'Lower7'
)
""").df()

print(
    "Lower-layer records beneath NO_CANOPY:",
    len(no_canopy_lower_violations)
)

if len(no_canopy_lower_violations):
    display(no_canopy_lower_violations.head(50))

assert len(no_canopy_lower_violations) == 0, (
    "Found lower-layer contacts beneath a __NO_CANOPY__ TopCanopy record. "
    "This violates the expected LPI recording structure and should be "
    "investigated before calculating composition."
)

print("NO_CANOPY protocol QA: PASS")


# A. Top-hit / top-down composition

The top-hit product represents the **top-down outcome at each pin**.

For each pin:

1. If `TopCanopy` contains an actual contact, that record is the top hit.
2. If `TopCanopy == "__NO_CANOPY__"`, then the pin has no lower-layer contacts by protocol and the **`SoilSurface` record becomes the top hit**.

`Lower1`–`Lower7` are never used to replace a `__NO_CANOPY__` observation.

The species and functional-group top-hit outputs remain **vegetation fractional cover only**:

- numerator = pins whose derived top hit is a plant taxon / functional group,
- denominator = **all sampled pins**.

Therefore a no-canopy pin whose SoilSurface is soil, litter, rock, lichen, etc. contributes zero to every plant species/FG cover while remaining in the denominator. This preserves true fractional cover rather than renormalizing only across vegetated pins.

A separate top-hit surface-composition QA table is also produced so the nonvegetated portion remains visible.


In [ ]:
# ============================================================================
# 7. DERIVE TOP HIT PER PIN + TOP-HIT SPECIES COVER
#
# Protocol:
#   - actual TopCanopy contact wins
#   - __NO_CANOPY__ falls directly to SoilSurface
#   - Lower1-Lower7 are not part of this derivation
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE lpi_top_hit AS

WITH top AS (

    SELECT
        PrimaryKey,
        Year,
        LineKey,
        PointNbr,

        observed_code,
        code_class,
        taxon_id,
        taxon_label,
        USDA_rank,
        MOSAIC_FG,

        CASE
            WHEN observed_code = '__NO_CANOPY__'
            THEN 1
            ELSE 0
        END AS no_canopy

    FROM lpi_valid

    WHERE layer = 'TopCanopy'
),

surface AS (

    SELECT
        PrimaryKey,
        Year,
        LineKey,
        PointNbr,

        observed_code,
        code_class,
        taxon_id,
        taxon_label,
        USDA_rank,
        MOSAIC_FG

    FROM lpi_valid

    WHERE layer = 'SoilSurface'
)

SELECT

    t.PrimaryKey,
    t.Year,
    t.LineKey,
    t.PointNbr,

    CASE
        WHEN t.no_canopy = 0
        THEN t.observed_code
        ELSE s.observed_code
    END AS observed_code,

    CASE
        WHEN t.no_canopy = 0
        THEN t.code_class
        ELSE s.code_class
    END AS code_class,

    CASE
        WHEN t.no_canopy = 0
        THEN t.taxon_id
        ELSE s.taxon_id
    END AS taxon_id,

    CASE
        WHEN t.no_canopy = 0
        THEN t.taxon_label
        ELSE s.taxon_label
    END AS taxon_label,

    CASE
        WHEN t.no_canopy = 0
        THEN t.USDA_rank
        ELSE s.USDA_rank
    END AS USDA_rank,

    CASE
        WHEN t.no_canopy = 0
        THEN t.MOSAIC_FG
        ELSE s.MOSAIC_FG
    END AS MOSAIC_FG,

    CASE
        WHEN t.no_canopy = 0
        THEN 'TopCanopy'
        ELSE 'SoilSurface'
    END AS top_hit_source

FROM top t

LEFT JOIN surface s
    USING (
        PrimaryKey,
        Year,
        LineKey,
        PointNbr
    )
;
""")

# Every sampled pin should resolve to one top-hit row.
top_hit_qa = con.execute("""
SELECT
    COUNT(*) AS n_top_hit_rows,
    COUNT(
        DISTINCT (
            PrimaryKey,
            Year,
            LineKey,
            PointNbr
        )
    ) AS n_unique_top_hit_pins,
    SUM(
        CASE
            WHEN observed_code IS NULL
            THEN 1
            ELSE 0
        END
    ) AS unresolved_top_hits
FROM lpi_top_hit
""").df()

display(top_hit_qa)

# Species fractional cover, denominator = all sampled pins.
con.execute("""
CREATE OR REPLACE TABLE top_species_long AS

WITH hits AS (

    SELECT
        PrimaryKey,
        Year,
        taxon_id,
        taxon_label,
        USDA_rank,
        COUNT(*) AS n_hit_points

    FROM lpi_top_hit

    WHERE
        code_class = 'Plant'
        AND taxon_id IS NOT NULL

    GROUP BY
        PrimaryKey,
        Year,
        taxon_id,
        taxon_label,
        USDA_rank
)

SELECT
    h.PrimaryKey,
    h.Year,
    h.taxon_id,
    h.taxon_label,
    h.USDA_rank,
    h.n_hit_points,
    p.n_points,
    100.0 * h.n_hit_points / p.n_points AS percent_cover

FROM hits h

JOIN plot_totals p
    USING (PrimaryKey, Year)
;
""")


In [ ]:
# ============================================================================
# 8. TOP-HIT FUNCTIONAL-GROUP COVER + SURFACE-COMPOSITION QA
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE top_fg_long AS

WITH hits AS (

    SELECT
        PrimaryKey,
        Year,
        MOSAIC_FG,
        COUNT(*) AS n_hit_points

    FROM lpi_top_hit

    WHERE
        code_class = 'Plant'
        AND MOSAIC_FG IS NOT NULL

    GROUP BY
        PrimaryKey,
        Year,
        MOSAIC_FG
)

SELECT
    h.PrimaryKey,
    h.Year,
    h.MOSAIC_FG,
    h.n_hit_points,
    p.n_points,
    100.0 * h.n_hit_points / p.n_points AS percent_cover

FROM hits h

JOIN plot_totals p
    USING (PrimaryKey, Year)
;
""")

# Plant FG fractional cover cannot exceed 100% because each pin has one top hit.
top_fg_qa = con.execute("""
SELECT
    PrimaryKey,
    Year,
    SUM(percent_cover) AS summed_top_plant_cover

FROM top_fg_long

GROUP BY
    PrimaryKey,
    Year

HAVING
    SUM(percent_cover) > 100.000001
""").df()

assert len(top_fg_qa) == 0

# Preserve nonvegetated top-down composition for QA / interpretation.
top_surface_composition = con.execute("""
SELECT

    PrimaryKey,
    Year,
    observed_code,
    code_class,
    top_hit_source,

    COUNT(*) AS n_points

FROM lpi_top_hit

GROUP BY
    PrimaryKey,
    Year,
    observed_code,
    code_class,
    top_hit_source

ORDER BY
    Year,
    PrimaryKey,
    n_points DESC
""").df()

TOP_SURFACE_QA_FILE = (
    QA_DIR /
    "LPI_top_hit_surface_composition_long.csv"
)

top_surface_composition.to_csv(
    TOP_SURFACE_QA_FILE,
    index=False
)

top_source_qa = con.execute("""
SELECT
    top_hit_source,
    COUNT(*) AS n_pins,
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER () AS percent_pins
FROM lpi_top_hit
GROUP BY top_hit_source
ORDER BY n_pins DESC
""").df()

display(top_source_qa)

print("Top-hit cover exclusivity QA: PASS")
print("\nTop-hit surface-composition QA:")
print(TOP_SURFACE_QA_FILE)


# ============================================================================
# 12. FIRST-HIT SURFACE / PROTOCOL QA
#
# All PlantBase-containing visits were excluded before composition processing.
# Therefore no retained first-hit row may contain PlantBase.
# ============================================================================

top_surface_composition = con.execute("""
SELECT
    PrimaryKey,
    Year,
    observed_code,
    canonical_code,
    code_class,
    dictionary_resolution_status,
    top_hit_source,
    COUNT(*) AS n_points
FROM lpi_top_hit
GROUP BY
    PrimaryKey,
    Year,
    observed_code,
    canonical_code,
    code_class,
    dictionary_resolution_status,
    top_hit_source
ORDER BY Year, PrimaryKey, n_points DESC
""").df()

TOP_SURFACE_QA_FILE = QA_DIR / "LPI_top_hit_surface_composition_long.csv"
top_surface_composition.to_csv(TOP_SURFACE_QA_FILE, index=False)

retained_plantbase = con.execute("""
SELECT COUNT(*)
FROM lpi_valid
WHERE code_class = 'PlantBase'
""").fetchone()[0]

assert retained_plantbase == 0, (
    "PlantBase hits remain in lpi_valid even though PlantBase visits "
    "should have been excluded."
)

print("Retained PlantBase hits:", retained_plantbase)


In [ ]:
# ============================================================================
# 9. MULTILAYER SPECIES COVER — LONG TABLE
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE multi_species_long AS

WITH presence AS (

    SELECT DISTINCT
        PrimaryKey,
        Year,
        LineKey,
        PointNbr,
        taxon_id,
        taxon_label,
        USDA_rank

    FROM lpi

    WHERE
        layer IN (
            'TopCanopy',
            'Lower1',
            'Lower2',
            'Lower3',
            'Lower4',
            'Lower5',
            'Lower6',
            'Lower7',
            'SoilSurface'
        )
        AND code_class = 'Plant'
        AND taxon_id IS NOT NULL
),

hits AS (

    SELECT
        PrimaryKey,
        Year,
        taxon_id,
        taxon_label,
        USDA_rank,
        COUNT(*) AS n_hit_points

    FROM presence

    GROUP BY
        PrimaryKey,
        Year,
        taxon_id,
        taxon_label,
        USDA_rank
)

SELECT
    h.PrimaryKey,
    h.Year,
    h.taxon_id,
    h.taxon_label,
    h.USDA_rank,
    h.n_hit_points,
    p.n_points,
    100.0 * h.n_hit_points / p.n_points AS percent_cover

FROM hits h

JOIN plot_totals p
    USING (PrimaryKey, Year)
;
""")


In [ ]:
# ============================================================================
# 10. MULTILAYER FUNCTIONAL-GROUP COVER — LONG TABLE
# ============================================================================

con.execute("""
CREATE OR REPLACE TABLE multi_fg_long AS

WITH presence AS (

    SELECT DISTINCT
        PrimaryKey,
        Year,
        LineKey,
        PointNbr,
        MOSAIC_FG

    FROM lpi_valid

    WHERE
        layer IN (
            'TopCanopy',
            'Lower1',
            'Lower2',
            'Lower3',
            'Lower4',
            'Lower5',
            'Lower6',
            'Lower7',
            'SoilSurface'
        )
        AND code_class = 'Plant'
        AND MOSAIC_FG IS NOT NULL
),

hits AS (

    SELECT
        PrimaryKey,
        Year,
        MOSAIC_FG,
        COUNT(*) AS n_hit_points

    FROM presence

    GROUP BY
        PrimaryKey,
        Year,
        MOSAIC_FG
)

SELECT
    h.PrimaryKey,
    h.Year,
    h.MOSAIC_FG,
    h.n_hit_points,
    p.n_points,
    100.0 * h.n_hit_points / p.n_points AS percent_cover

FROM hits h

JOIN plot_totals p
    USING (PrimaryKey, Year)
;
""")


In [ ]:
# ============================================================================
# 11. CONTEXT-DEPENDENT / UNRESOLVED QA
# ============================================================================

context_qa = con.execute("""
SELECT
    observed_code,
    code_class,
    layer,
    COUNT(*) AS n_records,
    COUNT(DISTINCT PrimaryKey) AS n_plot_visits,

    COUNT(
        DISTINCT (
            PrimaryKey,
            LineKey,
            PointNbr
        )
    ) AS n_points

FROM lpi_valid

WHERE
    code_class IN (
        'ContextDependent',
        'ProtocolUnresolved'
    )

GROUP BY
    observed_code,
    code_class,
    layer

ORDER BY
    n_records DESC
""").df()

display(context_qa)

context_qa.to_csv(
    QA_DIR / "LPI_context_dependent_unresolved.csv",
    index=False
)


# ============================================================================
# 15. UNRESOLVED-CODE QA BY YEAR — RETAINED VISITS ONLY
# ============================================================================

resolution_qa = con.execute("""
SELECT
    Year,
    dictionary_resolution_status,
    code_class,
    COUNT(*) AS n_records,
    COUNT(DISTINCT observed_code) AS n_codes,
    COUNT(DISTINCT PrimaryKey) AS n_plot_visits
FROM lpi_valid
WHERE dictionary_resolution_status IN (
    'UnknownPlant',
    'UnknownCode'
)
GROUP BY Year, dictionary_resolution_status, code_class
ORDER BY Year, n_records DESC
""").df()

display(resolution_qa)

resolution_qa.to_csv(
    QA_DIR / "LPI_resolution_tail_by_year.csv",
    index=False,
)


# Export year-specific composition matrices

Each output row is one `PrimaryKey × Year`. A true revisit with a different `PrimaryKey` remains a separate row.

Missing taxa/groups within a retained plot visit are written as `0.0`. Species columns use final `canonical_code`, not raw observed codes, so finalized synonym redirects aggregate correctly.

Every output also carries visit-level QA fields:

- `n_all_lpi_hits`
- `n_unknown_code_hits`
- `unknown_code_pct_all_hits`
- `unknown_code_gt5pct`

Visits containing `PlantBase` are absent from all four composition products.


In [ ]:
# ============================================================================
# 16. EXPORT FUNCTION — LONG COVER TABLE -> YEAR-SPECIFIC WIDE CSV
#
# Every retained plot visit is exported, including visits with zero plant
# cover for a particular ecological resolution.
# ============================================================================

META_COLS = [
    "PrimaryKey",
    "Year",
    "DateVisited",
    "DBKey",
    "ProjectKey",
    "source",
    "Latitude_NAD83",
    "Longitude_NAD83",
    "n_points",
    "n_all_lpi_hits",
    "n_unknown_code_hits",
    "unknown_code_pct_all_hits",
    "unknown_code_gt5pct",
]


def export_year_wide(
    long_table,
    category_col,
    prefix,
    output_dir,
    filename_stub,
):
    """Write one wide composition CSV per year."""

    table_years = [
        int(x[0])
        for x in con.execute(
            """
            SELECT DISTINCT Year
            FROM plot_metadata
            WHERE Year IS NOT NULL
            ORDER BY Year
            """
        ).fetchall()
    ]

    written = []

    for year in table_years:
        meta = con.execute(f"""
            SELECT
                {", ".join(META_COLS)}
            FROM plot_metadata
            WHERE Year = {year}
            ORDER BY PrimaryKey
        """).df()

        long_df = con.execute(f"""
            SELECT
                PrimaryKey,
                Year,
                {category_col},
                percent_cover
            FROM {long_table}
            WHERE Year = {year}
        """).df()

        if long_df.empty:
            wide = meta.copy()
        else:
            matrix = (
                long_df
                .pivot_table(
                    index=["PrimaryKey", "Year"],
                    columns=category_col,
                    values="percent_cover",
                    aggfunc="first",
                    fill_value=0.0,
                )
                .reset_index()
            )

            category_columns = [
                c for c in matrix.columns
                if c not in ["PrimaryKey", "Year"]
            ]

            matrix = matrix.rename(
                columns={c: f"{prefix}{c}" for c in category_columns}
            )

            wide = meta.merge(
                matrix,
                on=["PrimaryKey", "Year"],
                how="left",
                validate="one_to_one",
            )

            cover_cols = [c for c in wide.columns if c.startswith(prefix)]
            wide[cover_cols] = wide[cover_cols].fillna(0.0)

        assert not wide[["PrimaryKey", "Year"]].duplicated().any()

        outfile = output_dir / f"{filename_stub}_{year}.csv"
        wide.to_csv(outfile, index=False)

        n_flagged = int(
            wide["unknown_code_gt5pct"]
            .fillna(False)
            .astype(bool)
            .sum()
        )

        written.append({
            "year": year,
            "rows": len(wide),
            "columns": len(wide.columns),
            "flagged_unknown_gt5pct": n_flagged,
            "file": str(outfile),
        })

        print(
            f"{year}: {len(wide):,} rows × {len(wide.columns):,} columns; "
            f"{n_flagged:,} >5% unknown-code flags -> {outfile.name}"
        )

    return pd.DataFrame(written)


In [ ]:
# ============================================================================
# 14. WRITE TOP-HIT FUNCTIONAL-GROUP MATRICES BY YEAR
# ============================================================================

top_fg_files = export_year_wide(
    long_table="top_fg_long",
    category_col="MOSAIC_FG",
    prefix="FG_",
    output_dir=TOP_FG_DIR,
    filename_stub="LPI_top_hit_functional_group",
)

display(top_fg_files)


In [ ]:
# ============================================================================
# 18. CROSS-OUTPUT QA
# ============================================================================

expected_years = {
    int(x[0])
    for x in con.execute("""
        SELECT DISTINCT Year
        FROM plot_metadata
        WHERE Year IS NOT NULL
    """).fetchall()
}

year_sets = {
    "top_species": set(top_species_files["year"]),
    "top_fg": set(top_fg_files["year"]),
    "multi_species": set(multi_species_files["year"]),
    "multi_fg": set(multi_fg_files["year"]),
}

for name, year_set in year_sets.items():
    assert year_set == expected_years, (
        f"{name} year coverage differs from the retained LPI year universe."
    )

expected_rows = con.execute("""
SELECT
    Year AS year,
    COUNT(*) AS n_retained_visits
FROM plot_metadata
GROUP BY Year
ORDER BY Year
""").df()

for name, file_df in [
    ("top_species", top_species_files),
    ("top_fg", top_fg_files),
    ("multi_species", multi_species_files),
    ("multi_fg", multi_fg_files),
]:
    check = file_df.merge(expected_rows, on="year", how="left")
    assert (check["rows"] == check["n_retained_visits"]).all(), (
        f"{name} does not contain exactly one row per retained PrimaryKey × Year."
    )

multi_species_qa = con.execute("""
SELECT * FROM multi_species_long
WHERE percent_cover > 100.000001
""").df()

multi_fg_qa = con.execute("""
SELECT * FROM multi_fg_long
WHERE percent_cover > 100.000001
""").df()

assert len(multi_species_qa) == 0
assert len(multi_fg_qa) == 0

assert con.execute("""
SELECT COUNT(*)
FROM lpi_valid
WHERE code_class = 'PlantBase'
""").fetchone()[0] == 0

flag_mismatch = con.execute(f"""
SELECT *
FROM visit_unknown_code_qa
WHERE unknown_code_gt5pct
      <> (unknown_code_pct_all_hits > {UNKNOWN_CODE_FLAG_THRESHOLD_PCT})
""").df()

assert len(flag_mismatch) == 0

canon_qa = con.execute("""
SELECT
    observed_code,
    canonical_code,
    COUNT(*) AS n_records
FROM lpi_valid
WHERE observed_code IN ('POAR2R2', 'BAPRV', 'BAPRG')
GROUP BY observed_code, canonical_code
ORDER BY observed_code
""").df()

display(canon_qa)

for raw_code, expected in [
    ("POAR2R2", "PONIN"),
    ("BAPRV", "BAPR5"),
    ("BAPRG", "BAPR5"),
]:
    x = canon_qa.loc[canon_qa["observed_code"].eq(raw_code)]
    if len(x):
        assert set(x["canonical_code"]) == {expected}

print("Cross-output QA: PASS")
print("Output root:", OUTPUT_DIR)


# Resulting yearly data architecture

For each retained visit year the notebook writes exactly four composition families:

1. `top_hit/species/LPI_top_hit_species_<YEAR>.csv`
2. `top_hit/functional_group/LPI_top_hit_functional_group_<YEAR>.csv`
3. `multilayer/species/LPI_multilayer_species_<YEAR>.csv`
4. `multilayer/functional_group/LPI_multilayer_functional_group_<YEAR>.csv`

Each row is one `PrimaryKey × Year`; genuine revisits with different `PrimaryKey` values remain distinct.

Before composition is calculated, any visit containing `PlantBase` is excluded entirely.

All retained outputs include:

- `n_all_lpi_hits`
- `n_unknown_code_hits`
- `unknown_code_pct_all_hits`
- `unknown_code_gt5pct`

The >5% flag is diagnostic only: flagged visits are retained so they can be reviewed or filtered downstream without losing provenance.


In [ ]:
# ============================================================================
# 17. CROSS-OUTPUT QA
# ============================================================================

# Every output family should cover the same year universe.
year_sets = {
    "top_species": set(top_species_files["year"]),
    "top_fg": set(top_fg_files["year"]),
    "multi_species": set(multi_species_files["year"]),
    "multi_fg": set(multi_fg_files["year"]),
}

assert (
    year_sets["top_species"]
    == year_sets["top_fg"]
    == year_sets["multi_species"]
    == year_sets["multi_fg"]
)

# Top-hit plant FG sums cannot exceed 100%.
top_sum_qa = con.execute("""
SELECT
    PrimaryKey,
    Year,
    SUM(percent_cover) AS summed_cover
FROM top_fg_long
GROUP BY
    PrimaryKey,
    Year
HAVING SUM(percent_cover) > 100.000001
""").df()

assert len(top_sum_qa) == 0

# Individual multilayer taxa/groups cannot exceed 100%.
multi_species_qa = con.execute("""
SELECT *
FROM multi_species_long
WHERE percent_cover > 100.000001
""").df()

multi_fg_qa = con.execute("""
SELECT *
FROM multi_fg_long
WHERE percent_cover > 100.000001
""").df()

assert len(multi_species_qa) == 0
assert len(multi_fg_qa) == 0

print("Cross-output QA: PASS")

print("\nOutput root:")
print(OUTPUT_DIR)


# Resulting data architecture

For every year, the notebook produces four vegetation composition matrices:

### Top-hit species
`top_hit/species/LPI_top_hit_species_<YEAR>.csv`

Top-hit is the top-down outcome at each pin:

- actual `TopCanopy` contact when present,
- otherwise `SoilSurface` when `TopCanopy == "__NO_CANOPY__"`.

Only plant top hits contribute to species cover, but **all pins remain in the denominator**.

### Top-hit functional groups
`top_hit/functional_group/LPI_top_hit_functional_group_<YEAR>.csv`

Functional-group fractional cover uses the same top-hit definition and all-pin denominator.

### Multilayer species
`multilayer/species/LPI_multilayer_species_<YEAR>.csv`

A species is present at a pin if it occurs in any plant contact across the canopy profile or as a basal plant hit at `SoilSurface`.

### Multilayer functional groups
`multilayer/functional_group/LPI_multilayer_functional_group_<YEAR>.csv`

Functional-group presence is deduplicated directly at the pin level rather than obtained by summing species covers.

### Additional QA
`QA/LPI_top_hit_surface_composition_long.csv`

This preserves the nonvegetated top-down outcomes—soil, litter, rock, lichen, etc.—that occur when a no-canopy pin resolves to its SoilSurface observation.

The separation therefore gives two complementary ecological response spaces:

- **top-hit / top-down fractional cover** — closest to what an overhead optical sensor encounters,
- **multilayer / any-hit composition** — fuller vertically encountered plant community composition.
